[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/LTX-2-5_ComfyUI_Colab.ipynb) [View on GitHub](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/LTX-2-5_ComfyUI_Colab.ipynb)

**Open in Colab** ↑ for one-click run · **View on GitHub** ↑ for source



# 🎬 LTX-2.5 (ComfyUI runtime) — Text/Image-to-Video Generation

> **Runtime required:** GPU. **Recommended:** **L4 (24 GB)** for the FP8 distilled pipeline (≈99 s per short clip after VRAM purge). **T4 (16 GB)** works with NVFP4 + aggressive offload but is materially slower per-step (no native FP8/FP4 tensor cores on Turing).
>
> When Colab asks you to pick a runtime, choose **L4** if available; fall back to T4 if not. The notebook auto-detects the GPU.

A Colab port of [Lightricks LTX-2.5](https://huggingface.co/Lightricks/LTX-2.5) — a 22B parameter distilled text/image-to-video model that produces **video with synchronized audio**, supports text-to-video, image-to-video, and video extension. Built on the [Lightricks/ComfyUI-LTXVideo](https://github.com/Lightricks/ComfyUI-LTXVideo) custom node pack.

**Workflow (L4 config) — matches Lightricks' example_workflows/2.5/:**

```
UNETLoader (transformer, ≈20 GB int8-convrot distilled)
  ↓
CLIPLoader (gemma text encoder, ≈15 GB int8-convrot) + CLIPTextEncode
  OR GemmaAPITextEncode (free API, no local model load)
  ↓
LTXVConditioning (positive + negative prompts at canvas fps)
  ↓
EmptyLTXVLatentVideo (canvas dims, length frames)
  + LTXVEmptyLatentAudio (joint AV diffusion requires both latents)
  → LTXVConcatAVLatent (NestedTensor [video, audio])
  ↓
CFGGuider + KSamplerSelect (euler_ancestral) + ManualSigmas (9-sigma distilled)
  ↓
LTXVNormalizingSampler (8-step distilled, CFG=1; prevents overbaking)
  ↓ → LATENT (NestedTensor [video, audio])
LTXVSeparateAVLatent (splits NestedTensor into video + audio latents)
  ↓
LTXVTiledVAEDecode (video, 2x2 spatial tiles - ≈40% VRAM reduction)
LTXVAudioVAEDecode (audio, from audio_vae)
  ↓
CreateVideo (fuses A/V into one .mp4) → SaveVideo (writes to COMFY_DIR/output/)
```

**Optional I2V / FLF2V conditioning** adds `LTXVAddGuide` nodes between the conditioning
and the sampler, with `LoadImage` for each conditioning frame (first/last frame strengths
control how rigidly the model adheres to those frames).

**Why not guillaume127/LTX-2.5-FP8 (the @TIMES99 / the README's path)?** guillaume's FP8 distilation reads as a diffusion-only state_dict (no `model.diffusion_model.` prefix), so it loads via `UNETLoader` from `models/diffusion_models/`. That part works. **The bottleneck is Gemma + UNET together:** 14.32 GB + 21.87 GB = **36.19 GB**, more than 24 GB. guillaume's README recommends a VRAM Cleanup node or `--highvram`; ComfyUI v0.32 has neither (`--highvram` is launch-only and assumes 32+ GB total). For 24 GB Colab, the int8-convrot distilled checkpoint + LTXVConditioning (no Gemma) is the path that actually fits.

**For systems with 32+ GB VRAM:** the user can switch `TRANSFORMER = guillaume127/LTX-2.5-FP8` in STEP 2 to get the FP8 weights; the workflow nodes are unchanged. They will then see ≈99 s/clip as guillaume measured on RTX 4090.

**For enhanced prompts:** the cell 8 form widget `USE_PROMPT_ENHANCER` switches between two paths:

- **API path (recommended):** when `LTX_API_KEY` is set and `USE_PROMPT_ENHANCER=True`, the workflow uses `GemmaAPITextEncode` (provided by `Lightricks/ComfyUI-LTXVideo`). Sub-second response, no local LLM load, no extra VRAM. Get a free key at https://console.ltx.io.
- **Local path (default):** when either is unset, the workflow uses `CLIPLoader + CLIPTextEncode` with the bundled Gemma text encoder (≈2 GB VRAM, runs locally in ≈1-3 s per prompt).

Both paths are auto-selected at workflow build time in `_build_workflow()`. The API path also enables `enhance_prompt=True` so short prompts get expanded into detailed cinematic instructions before encoding — which the distilled model benefits from significantly.

## Companion components (in STEP 2)

Every LTX-2.5 workflow loads three supporting files in addition to the transformer:

**Text encoder** (`TEXT_ENCODER`): Gemma 4 12B with LTX's `text_embedding_projection` + `audio_projector` layers baked in. Two variants on `Lightricks/LTX-2.5/text_encoders/`:

```
gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors  15.4 GB  - default; int8-convrot quantized
gemma4-12b-with-proj-ltx-2.5-bf16.safetensors               26.3 GB  - full precision; for 32+ GB hardware only
```

**When is this file loaded?**

- **Local prompt path (default):** `CLIPLoader + CLIPTextEncode` reads this file. ≈15 GB stays on disk but is loaded into VRAM during prompt encoding (ComfyUI auto-unloads it after each prompt change, so peak VRAM is ≈22 GB during sampling, ≈25 GB during prompt encode — fits L4).
- **API prompt path:** when `LTX_API_KEY` is set, `GemmaAPITextEncode` replaces `CLIPTextEncode` entirely. The text encoder file is **not** loaded into VRAM. Peak VRAM drops to ≈22 GB (sampling only).

**LTXVConditioning** (the node that consumes the encoded text) was the alternative in ComfyUI v0.31, but it requires a pre-loaded CLIP object — so any path that uses LTXVConditioning ultimately needs the text encoder loaded first. The diagram above reflects the v0.3.2+ workflow shape.

**Video VAE** (`VIDEO_VAE`): decodes the latent frames back to pixels. Two variants:

```
ltx-2.5-video-vae-conv-bf16.safetensors  1.45 GB  - Conv variant; faster decode; default
ltx-2.5-video-vae-bf16.safetensors       1.47 GB  - DiffVAE variant; marginally higher fidelity
```

The two VAEs are nearly indistinguishable at 832x480 and below. The DiffVAE becomes worth switching to only at 1504x832+ where its reconstruction sharpness matters. **The audio VAE** (`ltx-2.5-audio-vae-bf16.safetensors`, 0.34 GB) is auto-loaded and not configurable — it's small enough to always be present.

**Recommended for L4 (24 GB):** TEXT_ENCODER = int8-convrot, VIDEO_VAE = Conv.
**Recommended for T4 (16 GB):** Same encoder, Conv VAE. Avoid the DiffVAE at T4 — it costs 0.02 GB but uses more decode CUDA time which can push you past session timeout.

## Before you run STEP 2 — HuggingFace authentication

`Lightricks/LTX-2.5` and `gemma4-12b-*` are **gated repos** on HuggingFace. STEP 2 will fail with `401 Unauthorized` unless you:

1. Visit each gated repo and click **"Agree and access repository"**:
   - Foundation Model: https://huggingface.co/Lightricks/LTX-2.5
   - IC-LoRA Upscaler (for Step 9): https://huggingface.co/Lightricks/LTX-2.5-22b-IC-LoRA-Pixel-Spatial-Upscaler
2. Create a free read-only token at https://huggingface.co/settings/tokens
3. Either add it as a Colab secret named `HF_TOKEN` (Tools > Secrets in the left sidebar), or paste it into the `HF_TOKEN_BELOW` form widget at the top of STEP 2.

**GGUF repos** (`realrebelai/LTX-2.5_GGUFs`, `Abiray/LTX-2.5-Distilled-GGUF`) are public and do NOT need a token. But the supporting files (text encoder, video/audio VAEs, latent upscaler) still come from the gated `Lightricks/LTX-2.5` repo and need a token — so for any pipeline you need a token. The text encoder file is only loaded by the optional Gemma-CLIP path; for the default `LTXVConditioning` workflow it stays on disk but never loads into VRAM.

## Two-stage workflow (optional)
For ≈40% more detail at the same output resolution (or same detail at 2× resolution), toggle `USE_TWO_STAGE = True` in the cell 8 form widgets. The workflow then runs:
1. **Stage 1** — 8-step distilled sample at base resolution (e.g. 960×544). Produces a structurally-correct low-res latent + audio.
2. **Stage 2** — 2× spatial upscale via `ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors` + 3-step refinement. Adds high-frequency detail at the upscaled resolution.
Cost: ≈25% longer per clip (≈125s vs ≈99s on L4 at 960×544). Same distill path, same CFG=1, same negative prompt.
Per-scene toggle in batch JSON: `"two_stage": true` (or `false`). Useful for A/B comparing single vs two-stage within a batch.
Requires `USE_UPSCALER = True` in STEP 2 (downloads the spatial upscaler, ≈1 GB).
## Video upscaler (STEP 9)
For upscaling existing videos (not generating from scratch), run STEP 9 after generating. It:
1. Loads the source mp4 via VHS (`VHS_LoadVideoPath`)
2. Extracts the first frame for continuity (`VHS_SplitImages` → `LTXVImgToVideoInplace`)
3. Spatial-upscales the latent 2× (or 4× via two chained passes) using `LTXVLatentUpsampler`
4. Runs a low-noise 4-step refinement pass with the joint LTX-2.5 diffusion model (sigma schedule derived from the STRENGTH slider)
5. Decodes the upscaled frames and muxes them with the **original audio** via `VHS_VideoCombine` (audio is passed through unchanged, not re-encoded through the LTX audio VAE)
**STRENGTH slider** controls how much the diffusion refines the source:
- 0.1–0.3 (default 0.3): subtle enhancement, source structure preserved
- 0.5: light cleanup (community-workflow default)
- 1.0: aggressive — significant generative regeneration
**UPSCALE_FACTOR** dropdown: 2× (one pass) or 4× (two chained 2× passes, ≈2× the runtime but cleaner).
The audio in your generated clips stays the same — STEP 9 only refines the pixels. So if you're using the audio in a music video, the upscaled mp4 keeps that exact audio track.
## GGUF quants (optional, lower VRAM)

Set `TRANSFORMER` in STEP 2 to one of the GGUF choices and pick a `QUANT` (Q2 through Q8):

```
realrebelai/LTX-2.5_GGUFs              Q2_K=8.83 GB, Q3_K_M=11.5, Q4_K_M=15.1, Q6_K=18.7, Q8_0=23.6
Abiray/LTX-2.5-Distilled-GGUF          Q3_K_S=12.6 (only here), Q4_K_M=15.7, Q6_K=18.6, Q8_0=23.6
ChrisColeTech/LTX-2.5-turbo-GGUF        (no auto-download; uses an existing GGUF in models/unet/ via CCTech custom loader)
```

GGUF files live in `models/unet/` (not `models/diffusion_models/`) and load via **Unet Loader (GGUF)** from the `city96/ComfyUI-GGUF` custom node pack (cloned in STEP 1). Same workflow nodes downstream (`EmptyLTXVLatentVideo`, `LTXVEmptyLatentAudio`, `LTXVConcatAVLatent`, `LTXVConditioning`, `CFGGuider`, `LTXVNormalizingSampler`, `LTXVSeparateAVLatent`, `LTXVTiledVAEDecode`, `LTXVAudioVAEDecode`, `CreateVideo`, `SaveVideo`).

**Pick GGUF when:**
- You have a 16 GB T4 and want LTX-2.5 to fit (Q2_K ≈9 GB dist).
- You have a 24 GB L4 but want to leave headroom for longer contexts/longer clips (Q4_K_M ≈15 GB vs 20 GB int8-convrot).
- You want the smallest Drive footprint (Q2_K uses ≈9 GB, less than half the int8-convrot dist).

**Stay on safetensors when:**
- You want maximum visual fidelity (Q8_0 ≈24 GB ≈ int8-convrot; not enough gain to switch).
- You want the built-in ComfyUI `UNETLoader` (no extra custom node).

**Tradeoffs:** GGUF quantization can subtly desync the audio stream at aggressive quants (`<Q4_K_M`) — the model card from realrebelai documents this and preserves `to_gate_logits` at higher precision to mitigate it. Q4_K_M and above are visually and audibly indistinguishable from bf16 for most prompts; Q3_K_M starts to lose fine detail. Q2_K is the smallest but the quality drop is visible.

## ⚠️ License

LTX-Video Open Weights License v0.1 — permits non-commercial research and personal use; commercial use requires a separate license. See [Lightricks/LTX-Video](https://huggingface.co/Lightricks/LTX-Video/blob/main/LTX-Video-Open-Weights-License-0.X.txt).







In [ ]:
#@title STEP 1 — Install ComfyUI + ComfyUI-LTXVideo + ComfyUI-GGUF (Drive-persistent)

"""
• Mounts Google Drive for the weights cache + ComfyUI installation
• Clones ComfyUI v0.30.1+ to /content/drive/MyDrive/AEI_ComfyUI/ (shared
  with MiniMax-H3 notebook — same install root, model-specific weights
  live in /content/drive/MyDrive/AEI_3D_Cache/LTX-Video-2.5/weights/)
• Installs torch 2.11.0+cu130 (L4 / T4 / A100 compatible)
• Installs ComfyUI's requirements.txt (transformers, tokenizers, safetensors, av, ...)
• Installs ComfyUI-LTXVideo's requirements.txt (diffusers, einops, ninja, ...)
• Clones Lightricks/ComfyUI-LTXVideo custom node pack into custom_nodes/
• Sets PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to reduce fragmentation
"""

import os, sys, subprocess, time, pathlib
from pathlib import Path

print('='*72)
print('LTX-2.5 / ComfyUI — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path('/content/_ltx_cache')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

COMFY_DIR = DRIVE_ROOT / 'AEI_ComfyUI'
HF_CACHE  = DRIVE_ROOT / 'AEI_3D_Cache' / 'LTX-Video-2.5'
OUT_DIR   = DRIVE_ROOT / 'AEI_3D_Out' / 'LTX-Video-2.5'
COMFY_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ComfyUI/custom_nodes/ might not exist yet - create it explicitly
# before any git clone so Drive FUSE doesn't fail the clone at the
# intermediate-dir step (exit 255 with empty stderr).
(COMFY_DIR / 'custom_nodes').mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME']               = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE']  = str(HF_CACHE)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.8')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print(f'  Drive cache  : {HF_CACHE}')
print(f'  ComfyUI dir  : {COMFY_DIR}')
print(f'  Output dir   : {OUT_DIR}')

# Ensure ComfyUI core is up to date (provides comfy_api for modern node packs)
_needs_comfy_sync = not COMFY_DIR.joinpath('main.py').exists() or not (COMFY_DIR / 'comfy_api').exists()
if _needs_comfy_sync:
    print(f'  Syncing ComfyUI core (v0.32+ with comfy_api) to {COMFY_DIR} ...')
    _tmp_comfy = Path('/content/_tmp_comfy_core')
    if _tmp_comfy.exists():
        import shutil as _sh
        _sh.rmtree(_tmp_comfy, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/comfyanonymous/ComfyUI.git', str(_tmp_comfy)], check=True)
    COMFY_DIR.mkdir(parents=True, exist_ok=True)
    for _item in _tmp_comfy.iterdir():
        if _item.name in ('.git', 'models', 'output', 'input', 'custom_nodes'):
            continue
        _dst = COMFY_DIR / _item.name
        if _item.is_dir():
            if _dst.exists():
                import shutil as _sh
                _sh.rmtree(_dst, ignore_errors=True)
            import shutil as _sh
            _sh.copytree(_item, _dst)
        else:
            import shutil as _sh
            _sh.copy2(_item, _dst)
    import shutil as _sh
    _sh.rmtree(_tmp_comfy, ignore_errors=True)
    print('  ✓ ComfyUI core updated to latest master!')
else:
    print(f'  Reusing existing {COMFY_DIR} (comfy_api verified)')


# Optional update block. Set UPDATE_COMFYUI = True only if you need to pull
# the latest ComfyUI commits. Uses a strict 15s timeout to prevent Drive FUSE hangs.
UPDATE_COMFYUI = False  #@param {type:"boolean"}
if UPDATE_COMFYUI:
    _upd_targets = [
        (str(COMFY_DIR), "ComfyUI"),
        (str(COMFY_DIR / 'custom_nodes' / 'ComfyUI-LTXVideo'), "ComfyUI-LTXVideo"),
        (str(COMFY_DIR / 'custom_nodes' / 'ComfyUI-GGUF'), "ComfyUI-GGUF"),
        (str(COMFY_DIR / 'custom_nodes' / 'ComfyUI-VideoHelperSuite'), "ComfyUI-VideoHelperSuite"),
    ]
    os.environ['GIT_TERMINAL_PROMPT'] = '0'
    for _d, _name in _upd_targets:
        if os.path.exists(os.path.join(_d, '.git')):
            print(f"  Updating {_name} ...", flush=True)
            try:
                _r = subprocess.run(['git', 'pull', '--ff-only'], cwd=_d,
                                    capture_output=True, text=True, timeout=15)
                if _r.returncode == 0:
                    print(f'    {_r.stdout.strip()[:200] or "Already up to date."}')
                else:
                    print(f'    Notice: {_r.stderr.strip()[:200]}')
            except subprocess.TimeoutExpired:
                print(f'    Timed out checking {_name} on Drive FUSE — continuing.')
            except Exception as _e:
                print(f'    Skipped {_name}: {type(_e).__name__}')
        else:
            print(f'  Skipping update for {_name} (no .git at {_d})')

# If LTX-2.5 audio VAE decoded fine but you started hitting
# RuntimeError: tensor a (3328) must match tensor b (128), the
# AudioLatentNormalizer in ComfyUI changed. Re-run STEP 1 with
# _update_block = True to get the latest patch; if the issue
# persists, try switching to the safetensors int8-convrot path
# (more battle-tested than the GGUF path).


print('  Installing pytorch cu130 ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'torch', 'torchvision', 'torchaudio',
                '--index-url', 'https://download.pytorch.org/whl/cu130'], check=False)

# tqdm is used by huggingface_hub's snapshot_download progress bar; without
# it, downloads quietly print "Downloading: 100%" with no rate/ETA. Install
# it explicitly so the 20 GB transformer pull doesn't go dark for 4-8 min.
print('  Installing tqdm + hf_transfer for progress bars ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'tqdm', 'hf_transfer'], check=False)
# HF_XET_HIGH_PERFORMANCE is the 2026 replacement for HF_HUB_ENABLE_HF_TRANSFER.
# We don't use snapshot_download/xet in STEP 2 anymore (direct downloads),
# but setting this avoids the deprecation warning when huggingface_hub
# inspects the env vars on import.
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'

print('  Installing ComfyUI requirements ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '-r', str(COMFY_DIR / 'requirements.txt')], check=False)

# ComfyUI-LTXVideo's requirements pulls diffusers + einops + ninja +
# transformers>=4.50 (Gemma 3 needs newer transformers). Install
# unconditionally since pip is idempotent.
print('  Installing ComfyUI-LTXVideo requirements ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'diffusers', 'einops', 'kornia',
                'transformers[timm]>=4.50.0',
                'huggingface_hub>=0.25.2'], check=False)

# gguf package - city96/ComfyUI-GGUF needs this to parse .gguf
# tensors. Renamed from gguf-python; published as gguf on PyPI.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'gguf>=0.10.0'], check=False)


# Helper for installing repos reliably onto Google Drive FUSE without git locking (exit 255)
import shutil as _sh

def safe_install_node_repo(repo_url, target_dir):
    t_path = Path(target_dir)
    tmp_path = Path(f'/content/_tmp_{t_path.name}')
    if tmp_path.exists():
        _sh.rmtree(tmp_path, ignore_errors=True)
    print(f'  Cloning {t_path.name} to local fast SSD ...', flush=True)
    subprocess.run(['git', 'clone', '--depth=1', repo_url, str(tmp_path)], check=True)
    
    t_path.mkdir(parents=True, exist_ok=True)
    for _item in tmp_path.iterdir():
        if _item.name == '.git':
            continue
        _dst = t_path / _item.name
        if _item.is_dir():
            if _dst.exists():
                _sh.rmtree(_dst, ignore_errors=True)
            _sh.copytree(_item, _dst)
        else:
            _sh.copy2(_item, _dst)
    _sh.rmtree(tmp_path, ignore_errors=True)
    # Patch pyramid_blending.py pad import if present (fixes kornia 0.8+ compatibility)
    _pyr = t_path / 'pyramid_blending.py'
    if _pyr.exists():
        _ptext = _pyr.read_text(encoding='utf-8')
        if 'from kornia.geometry.transform.pyramid import' in _ptext and 'pad' in _ptext:
            _ptext = _ptext.replace('from kornia.geometry.transform.pyramid import (
    pad,', 'from torch.nn.functional import pad\nfrom kornia.geometry.transform.pyramid import (')
            _ptext = _ptext.replace('from kornia.geometry.transform.pyramid import (', 'from torch.nn.functional import pad\nfrom kornia.geometry.transform.pyramid import (')
            _ptext = _ptext.replace('    pad,
', '')
            _pyr.write_text(_ptext, encoding='utf-8')

    _req = t_path / 'requirements.txt'
    if _req.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        '-r', str(_req)], check=False)
    print(f'  ✓ Installed {t_path.name} to Drive.')

# 1. ComfyUI-LTXVideo (Lightricks official custom node pack)
_LTX_DIR = COMFY_DIR / 'custom_nodes' / 'ComfyUI-LTXVideo'
safe_install_node_repo('https://github.com/Lightricks/ComfyUI-LTXVideo.git', _LTX_DIR)

# 2. city96/ComfyUI-GGUF
_GGUF_DIR = COMFY_DIR / 'custom_nodes' / 'ComfyUI-GGUF'
if not _GGUF_DIR.exists() or not (_GGUF_DIR / '__init__.py').exists():
    safe_install_node_repo('https://github.com/city96/ComfyUI-GGUF.git', _GGUF_DIR)
else:
    print(f'  Reusing existing {_GGUF_DIR.name}')

# 3. Kosinkadink/ComfyUI-VideoHelperSuite
_VHS_DIR = COMFY_DIR / 'custom_nodes' / 'ComfyUI-VideoHelperSuite'
if not _VHS_DIR.exists() or not (_VHS_DIR / '__init__.py').exists():
    safe_install_node_repo('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git', _VHS_DIR)
else:
    print(f'  Reusing existing {_VHS_DIR.name}')

# 4. Bambushu/redetail
_REDETAIL_DIR = COMFY_DIR / 'redetail'
if not _REDETAIL_DIR.exists() or not (_REDETAIL_DIR / 'redetail.py').exists():
    safe_install_node_repo('https://github.com/Bambushu/redetail.git', _REDETAIL_DIR)
else:
    print(f'  Reusing existing {_REDETAIL_DIR.name}')

# Link comfyui_cond_cache custom node into ComfyUI/custom_nodes/
_COND_CACHE_SRC = _REDETAIL_DIR / 'tools' / 'comfyui_cond_cache'
_COND_CACHE_DST = COMFY_DIR / 'custom_nodes' / 'comfyui_cond_cache'
if _COND_CACHE_SRC.exists() and not _COND_CACHE_DST.exists():
    try:
        _COND_CACHE_DST.symlink_to(_COND_CACHE_SRC)
        print(f'  Linked comfyui_cond_cache to {_COND_CACHE_DST}')
    except Exception:
        import shutil as _sh
        _sh.copytree(_COND_CACHE_SRC, _COND_CACHE_DST)
        print(f'  Copied comfyui_cond_cache to {_COND_CACHE_DST}')

import torch
print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda})')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU          : {p.name}  ({p.total_memory / 1024**3:.1f} GB)')
    print(f'  Compute      : {p.major}.{p.minor}')
else:
    raise SystemExit('No GPU detected - ComfyUI needs CUDA.')



In [ ]:
#@title STEP 2 — Download LTX-2.5 weights to Drive cache

"""
• Downloads based on the TRANSFORMER dropdown choice:
    - Lightricks/LTX-2.5 (int8-convrot distilled, 20 GB) — default; works on 24 GB L4
    - Lightricks/LTX-2.5 (nvfp4 distilled, 17 GB) — slightly tighter VRAM
    - guillaume127/LTX-2.5-FP8 (FP8, 21.87 GB) — keeps stock UNETLoader
    - realrebelai/LTX-2.5_GGUFs (Q2..Q8 GGUF distilled, 8.83..23.6 GB)
    - Abiray/LTX-2.5-Distilled-GGUF (Q2..Q8 GGUF distilled, adds Q3_K_S)
    - ChrisColeTech/LTX-2.5-turbo-GGUF — no auto-fetch; user supplies GGUF
      in models/unet/ (drop via ComfyUI Manager or a separate download)
• Common supporting files for all choices:
    - ltx-2.5-video-vae-conv-bf16 (video VAE, 1.35 GB)
    - ltx-2.5-audio-vae-bf16 (audio VAE, 0.34 GB) — placed in
      models/vae/ (matches the official Lightricks workflow placement).
      Stock ComfyUI VAELoader reads from there and correctly handles the
      audio_vae.* + vocoder.* state-dict prefixes via
      comfy/sd.py:VAE.__init__'s prefix detection. The audio VAE has
      joint diffusion with the video; LTXVEmptyLatentAudio creates
      the audio latent alongside EmptyLTXVLatentVideo before sampling.
    - ltx-2.5-latent-spatial-upscaler-x2 (0.93 GB; still required for the
      2.5 distilled pipeline per Lightricks release notes)
    - (optional) gemma4-12b-with-proj-ltx-2.5 text encoder (14 GB) — only
      needed if you swap LTXVConditioning for GemmaCLIPModelLoader +
      GemmaEnhancePrompt for LLM-enhanced prompts.
• Resolves expected sizes from HF manifest; HEAD-fallback for files
  where the manifest leaves size=None (older huggingface_hub versions).
• Per-repo fetches, then symlinks into ComfyUI/models/{diffusion_models,unet,vae}.
  Safetensors transformers (Lightricks/int8, nvfp4, guillaume FP8) live in
  models/diffusion_models/ and load via UNETLoader. GGUF transformers (any
  *_GGUF choice) live in models/unet/ and load via Unet Loader (GGUF) from
  city96/ComfyUI-GGUF (cloned in STEP 1).
"""
import os, sys, time, subprocess, urllib.request
from pathlib import Path

HF_CACHE = Path('/content/drive/MyDrive/AEI_3D_Cache/LTX-Video-2.5')
HF_CACHE.mkdir(parents=True, exist_ok=True)

# Weights selection — mode toggle chooses the transformer file. The
# text encoder + VAEs + upscaler are model-level (one per model, not
# per-mode). Same pattern as the MiniMax-H3 notebook's STEPS-default
# mode-routing (per-mode constants resolved to literals at build time,
# Colab's form parameter parser requires Python expressions).
# Default 'Lightricks int8-convrot' — the only LTX checkpoint that fits
# cleanly into the standard UNETLoader + VAELoader workflow AND works on
# 24 GB Colab cards (the FP8 and nvfp4 variants require either --highvram
# or a non-existent VRAM-cleanup node to avoid PCIe thrashing on 24 GB).
# User can override with 'guillaume127/LTX-2.5-FP8' once they verify their
# hardware can fit text encoder + transformer + VAE in 32+ GB.
# Select the diffusion transformer. Default is Lightricks distilled int8-convrot
# (fits 24 GB Colab with stock Gemma + Conv VAE). GGUF options use
# Llama.cpp-style quantization (Q2..Q8) loaded via 'Unet Loader (GGUF)'
# from city96/ComfyUI-GGUF for lower VRAM. Note: Python list literals cannot span lines.
TRANSFORMER = 'Lightricks/LTX-2.5 (distilled int8-convrot)'  #@param ["Lightricks/LTX-2.5 (distilled int8-convrot)", "Lightricks/LTX-2.5 (distilled nvfp4)", "guillaume127/LTX-2.5-FP8", "realrebelai/LTX-2.5_GGUFs (distilled GGUF)", "Abiray/LTX-2.5-Distilled-GGUF (distilled GGUF)", "ChrisColeTech/LTX-2.5-turbo-GGUF (uses Lightricks distilled GGUF)"] {"allow-input": true}
# Quant level for the GGUF options. Ignored for the int8-convrot,
# nvfp4, and FP8 safetensors transformers. Q4_K_M is the most
# common recommendation (best size/quality tradeoff); Q2_K is
# the smallest (~9 GB, fits 16 GB T4 if you disable most VAEs).
QUANT = "Q4_K_M"  #@param ["Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q4_K_S", "Q3_K_M", "Q2_K"] {"allow-input": true}
# Text encoder (Gemma 4 12B with LTX custom projection layers).
#   int8-convrot = 15.4 GB  - quantized for low VRAM; default
#   bf16         = 26.3 GB  - full precision; for 32+ GB hardware
# Most users should leave this at 'int8-convrot'. The bf16 variant
# only matters if you have 32+ GB and want marginally better prompt
# adherence (rarely worth the 11 GB extra). NOTE: this text encoder
# is only loaded if the workflow uses 'LTXVGemmaCLIPModelLoader' (the
# LLM-prompt-expansion path). The default workflow uses
# LTXVConditioning which does NOT load this file, so for the
# standard 24 GB path this dropdown has no runtime cost.
TEXT_ENCODER = "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot"  #@param ["gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot (int8 + convrot, 15.4 GB, default for 24 GB)", "gemma4-12b-with-proj-ltx-2.5-bf16 (bf16, 26.3 GB, for 32+ GB)"] {"allow-input": true}
TEXT_ENCODER = TEXT_ENCODER.split(" (")[0]
# Video VAE (decodes the latent -> pixels for video frames).
#   Conv = convolutional variant, 1.45 GB - faster decode, default
#   bf16 = DiffVAE,              1.47 GB - marginally higher fidelity
# The two variants are nearly indistinguishable on 832x480 output;
# the Conv VAE is ~10-15% faster at decode. The DiffVAE becomes
# worth switching to only at the highest resolutions (1504x832 and up).
VIDEO_VAE = "ltx-2.5-video-vae-conv-bf16"  #@param ["ltx-2.5-video-vae-conv-bf16 (Conv VAE, 1.45 GB, recommended)", "ltx-2.5-video-vae-bf16 (DiffVAE, 1.47 GB, marginally higher fidelity)"] {"allow-input": true}
VIDEO_VAE = VIDEO_VAE.split(" (")[0]

# Derive TEXT_ENCODER_FILE early (before any builtins export) so
# the export at the top of this cell can read it. (Without
# this reordering, the export block tried to assign
# builtins._TEXT_ENCODER_FILE before TEXT_ENCODER_FILE was defined.)
TEXT_ENCODER_FILE = TEXT_ENCODER + '.safetensors'


# Persist the dropdown values to builtins so STEPS 3 + 6 + 7 can read them.
# Done EARLY (right after widgets) so the export runs even if a later
# step (the HF download loop, the symlink loop, etc.) hits an error
# and the cell terminates with an exception. Without this, the user
# would need to fully re-run STEP 2 every time they hit any error.
import builtins as _b
_b._TRANSFORMER_VAR = TRANSFORMER
_b._QUANT_VAR = QUANT
print(f"  Loader config persisted to builtins early: TRANSFORMER={TRANSFORMER}, QUANT={QUANT}")

USE_UPSCALER = True  #@param {type:"boolean"}
print('='*72)
print('LTX-2.5 / ComfyUI — Download weights')
print('='*72)
print(f'  Transformer   : {TRANSFORMER}')
print(f'  Text encoder  : {TEXT_ENCODER}')
print(f'  Video VAE     : {VIDEO_VAE}')
print(f'  Spatial upscaler : {USE_UPSCALER}')
print()

import os
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_CACHE)

# HF_TOKEN: required for gated repos (Lightricks/LTX-2.5, guillaume127 FP8
# sometimes) but unnecessary for public repos (realrebelai / Abiray GGUF,
# guillaume's GGUF re-uploads). Read first from Colab secrets, fall back to
# the form widget below. Either is fine; secrets are slightly more secure.
os.environ.pop("HF_TOKEN", None)
_tok = None
_tok_source = None
try:
    from google.colab import userdata as _colab_userdata
    # Try common secret names - users pick one of three conventions:
    #   HF_TOKEN (recommended, matches Colab secrets convention)
    #   HUGGINGFACE_TOKEN (matches HF's CLI env var)
    #   HUGGINGFACE_HUB_TOKEN (matches HF SDK env var)
    for _secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_HUB_TOKEN", "HUGGINGFACE_API_KEY"):
        try:
            _tok = _colab_userdata.get(_secret_name)
            if _tok:
                _tok_source = _secret_name
                break
        except Exception:
            continue
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        print(f"  HF_TOKEN loaded from Colab secrets ({len(_tok)} chars, source={_tok_source!r})")
except Exception:
    pass
if not os.environ.get("HF_TOKEN"):
    print("  NOTE: no HF_TOKEN set. Lightricks/LTX-2.5 + most text encoders + VAEs")
    print("        are GATED on HuggingFace - downloads will 401 unless you've")
    print("        accepted the license on each gated repo and have a token.")
    print("        Steps to fix:")
    print("          1. Visit https://huggingface.co/Lightricks/LTX-2.5 and click")
    print("             'Agree and access repository'. Repeat for any other gated")
    print("             repo you want (guillaume127/LTX-2.5-FP8, gemma4-12b-*).")
    print("          2. Create a free read-token at https://huggingface.co/settings/tokens")
    print("          3. Either paste it into HF_TOKEN_BELOW below, OR add it as a")
    print("             Colab secret named HF_TOKEN (Tools > Secrets in the left")
    print("             sidebar, then 'Add secret'). The cell re-runs at the start")
    print("             of STEP 2 so secrets are picked up automatically.")
    print("        Workaround if you don't want to set up a token: switch")
    print("        TRANSFORMER to 'realrebelai/LTX-2.5_GGUFs' or 'Abiray/LTX-2.5-")
    print("        Distilled-GGUF' (transformer is public) but the supporting")
    print("        text encoder + VAEs + upscaler files STILL come from the gated")
    print("        Lightricks repo and will need a token anyway.")
    HF_TOKEN_BELOW = ""  #@param {type:"string"}
    if HF_TOKEN_BELOW.strip():
        os.environ["HF_TOKEN"] = HF_TOKEN_BELOW.strip()
        print(f"  HF_TOKEN set from form widget ({len(HF_TOKEN_BELOW)} chars)")

# Now that we may have an HF_TOKEN, snapshot_download and the direct
# /resolve/main/ endpoint should both work. Authorization Bearer header
# gets attached automatically by huggingface_hub (and by urllib with the
# Authorization env var trick below for our direct-download helper).


from huggingface_hub import HfApi
# NOTE: snapshot_download uses the xet CDN which 401s for anonymous
# callers from Colab IPs. We bypass it with _download_file() which
# uses the regular /resolve/main/ URL that does not require auth.
_RANGES_OK = True  # try to use range requests for resumability

# Map the user-friendly TRANSFORMER dropdown to a concrete filename +
# the upstream repo. The third option (Lightricks nvfp4) is on the
# same Lightricks/LTX-2.5 repo — just a different filename.
_TRANSFORMER_PATHS = {
    'Lightricks/LTX-2.5 (distilled int8-convrot)':
        ('Lightricks/LTX-2.5',
         'diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors'),
    'Lightricks/LTX-2.5 (distilled nvfp4)':
        ('Lightricks/LTX-2.5',
         'diffusion_models/ltx-2.5-22b-distilled-transformer-nvfp4.safetensors'),
    'guillaume127/LTX-2.5-FP8':
        ('guillaume127/LTX-2.5-FP8',
         'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors'),
}
# GGUF transformer metadata: (repo_id, filename_in_repo, scope_path).
# The filename varies by QUANT; scope_path is 'unet' for GGUF
# (city96 loader convention) or 'diffusion_models' for safetensors.
_GGUF_FILES = {
    'realrebelai/LTX-2.5_GGUFs': {
        'Q8_0':    'LTX-2.5-Distilled-Q8_0.gguf',
        'Q6_K':    'LTX-2.5-Distilled-Q6_K.gguf',
        'Q5_K_M':  'LTX-2.5-Distilled-Q5_K_M.gguf',
        'Q4_K_M':  'LTX-2.5-Distilled-Q4_K_M.gguf',
        'Q4_K_S':  'LTX-2.5-Distilled-Q4_K_S.gguf',
        'Q3_K_M':  'LTX-2.5-Distilled-Q3_K_M.gguf',
        'Q2_K':    'LTX-2.5-Distilled-Q2_K.gguf',
    },
    'Abiray/LTX-2.5-Distilled-GGUF': {
        'Q8_0':    'LTX-2.5-Distilled-Q8_0.gguf',
        'Q6_K':    'LTX-2.5-Distilled-Q6_K.gguf',
        'Q5_K_M':  'LTX-2.5-Distilled-Q5_K_M.gguf',
        'Q4_K_M':  'LTX-2.5-Distilled-Q4_K_M.gguf',
        'Q4_K_S':  'LTX-2.5-Distilled-Q4_K_S.gguf',
        'Q3_K_M':  'LTX-2.5-Distilled-Q3_K_M.gguf',
        # Abiray ships Q3_K_S too — bonus over realrebelai.
        'Q3_K_S':  'LTX-2.5-Distilled-Q3_K_S.gguf',
    },
    'ChrisColeTech/LTX-2.5-turbo-GGUF': {
        # Repo currently holds VAE + upscaler split files but no
        # transformer as of 2026-08-12. We don't auto-download from
        # ChrisColeTech (nothing transformer-shaped there). If the
        # user picks this TRANSFORMER, they need to have already
        # dropped a GGUF (e.g. Q4_K_M from realrebelai) into
        # models/unet/ via a prior run. The 'filename' here is the
        # one city96-GGUF-style loader expects.
        'Q8_0':    'LTX-2.5-Distilled-Q8_0.gguf',
        'Q6_K':    'LTX-2.5-Distilled-Q6_K.gguf',
        'Q5_K_M':  'LTX-2.5-Distilled-Q5_K_M.gguf',
        'Q4_K_M':  'LTX-2.5-Distilled-Q4_K_M.gguf',
        'Q4_K_S':  'LTX-2.5-Distilled-Q4_K_S.gguf',
        'Q3_K_M':  'LTX-2.5-Distilled-Q3_K_M.gguf',
        'Q3_K_S':  'LTX-2.5-Distilled-Q3_K_S.gguf',
        'Q2_K':    'LTX-2.5-Distilled-Q2_K.gguf',
    },
}
# Which ComfyUI loader class to use, and whether the file is
# GGUF (load via 'Unet Loader (GGUF)' from city96, scope=unet) or
# safetensors (load via stock UNETLoader, scope=diffusion_models).
_LOADER_BY_TRANSFORMER = {
    'realrebelai/LTX-2.5_GGUFs (distilled GGUF)':
        ('realrebelai/LTX-2.5_GGUFs', 'unet', 'UnetLoaderGGUF'),
    'Abiray/LTX-2.5-Distilled-GGUF (distilled GGUF)':
        ('Abiray/LTX-2.5-Distilled-GGUF', 'unet', 'UnetLoaderGGUF'),
    'ChrisColeTech/LTX-2.5-turbo-GGUF (uses Lightricks distilled GGUF)':
        # Same repo ID for fetch as realrebelai (CCTech has no
        # transformer file currently) but loader kind is CCTech's
        # custom node. Falls back to 'UnetLoaderGGUF' if CCTech's
        # custom node isn't installed.
        ('realrebelai/LTX-2.5_GGUFs', 'unet', 'CCTechUnetLoader'),
}
_loader = _LOADER_BY_TRANSFORMER.get(TRANSFORMER)
if _loader:
    _gg_repo, _scope, _loader_class = _loader
    if TRANSFORMER.startswith('ChrisColeTech'):
        # CCTech repo has no transformer; use realrebelai's file
        # but reference the CCTech loader (node class). The user
        # is responsible for installing the loader via ComfyUI Manager.
        _t_repo = _gg_repo
        _t_filename = _GGUF_FILES['ChrisColeTech/LTX-2.5-turbo-GGUF'][QUANT]
    else:
        _t_repo = _gg_repo
        # _GGUF_FILES keys are repo-id bases (not the dropdown label)
        _base_repo_key = TRANSFORMER.split(' (')[0]
        _t_filename = _GGUF_FILES[_base_repo_key][QUANT]
else:
    _t_repo, _t_filename = _TRANSFORMER_PATHS[TRANSFORMER]
    _scope = 'diffusion_models'
    _loader_class = 'UNETLoader'
print(f'  Resolving sizes from HF manifests ...')
_expected = {}
# Resolve HF manifest sizes for every repo we might pull from.
_repos_to_query = {
    _t_repo,
    'Lightricks/LTX-2.5',
    'guillaume127/LTX-2.5-FP8',
    'realrebelai/LTX-2.5_GGUFs',
    'Abiray/LTX-2.5-Distilled-GGUF',
    'ChrisColeTech/LTX-2.5-turbo-GGUF',
}
# Resolve HF manifest sizes. This runs BEFORE the file downloads
# (it's a quick API call), so the local cache for each repo may not
# exist yet - that's why we suppress FileNotFoundError here. If size
# resolution fails, the download still works (we just lose the
# 'expected vs actual' verification).
print(f'  Resolving sizes from HF manifests ...')
for _repo in _repos_to_query:
    try:
        # Use HF_TOKEN if set, so we can query the gated Lightricks/LTX-2.5
        # repo for file sizes. Without a token the API returns 401 but we
        # still try and tolerate the failure.
        _tok = os.environ.get('HF_TOKEN')
        info = HfApi(token=_tok).repo_info(_repo, files_metadata=True)
        for sib in info.siblings:
            if sib.size is not None:
                _expected[(_repo, sib.rfilename)] = sib.size
    except Exception as _info_err:
        # Quietly skip - downloads work without manifest sizes, the
        # print below still shows the canonical ComfyUI tree state.
        pass

def _size(repo, fn, fallback_path=None):
    sz = _expected.get((repo, fn))
    if sz is not None:
        return sz
    print(f'  size missing in manifest; HEAD-resolving {repo}/{fn} ...')
    url = f'https://huggingface.co/{repo}/resolve/main/{fn}'
    if fallback_path:
        url = f'https://huggingface.co/{repo}/resolve/main/{fallback_path}/{fn}'
    req = urllib.request.Request(url, method='HEAD')
    with urllib.request.urlopen(req, timeout=30) as r:
        return int(r.headers['content-length'])

# Bypass snapshot_download's xet-CDN path entirely. Direct
# request to huggingface.co/<repo>/resolve/main/<path> works
# for both .safetensors and .gguf files without a token.
# Resumable via HTTP Range; prints a progress line every 100 MB.
def _download_file(repo_id, filename, dest_path):
    """Download a single file from HF Hub with comprehensive error handling.

    Uses hf_hub_download (the canonical HF-supplied path) so we get
    proper exception types for gated-vs-private-vs-missing-file
    disambiguation. Falls back to a urllib GET only as a last resort
    if hf_hub_download is unavailable.

    The destination path is computed by the caller based on the chosen
    scope (diffusion_models, vae, unet, etc.). We let HF keep its own
    internal cache at HF_CACHE; symlink_dest (when supported) lets
    both location tracking AND content hashing work.
    """
    if Path(dest_path).exists():
        # Already at the expected local path. Skip.
        return str(dest_path)
    # If the parent has a partial .part file (older runs left them
    # behind), clear it - hf_hub_download will re-validate via ETag.
    _part = Path(str(dest_path) + '.part')
    if _part.exists():
        try:
            _part.unlink()
        except OSError:
            pass

    # Determine repo_type and subfolder for accurate HF resolution.
    _subfolder = None
    _filename = filename
    if '/' in filename:
        _subfolder, _filename = filename.rsplit('/', 1)

    _tok = os.environ.get('HF_TOKEN')
    try:
        from huggingface_hub import hf_hub_download
        _local = hf_hub_download(
            repo_id=repo_id,
            filename=_filename,
            subfolder=_subfolder,
            repo_type='model',
            local_dir=str(HF_CACHE),
            token=_tok or None,
            force_download=False,  # use ETag
        )
        # hf_hub_download with local_dir places file at the same relative
        # path as in the repo. If the caller wanted it at a specific path
        # (e.g., scope_dispatch), symlink there if it doesn't exist.
        if str(_local) != str(dest_path):
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            if not dest_path.exists():
                try:
                    dest_path.symlink_to(_local)
                except (OSError, NotImplementedError):
                    # Fall back to hard copy on filesystems that don't
                    # support symlinks (Drive FUSE on Colab IS supported,
                    # but be defensive).
                    import shutil
                    shutil.copy(_local, dest_path)
        return str(dest_path if dest_path.exists() else _local)
    except Exception as _e:
        # Catch generic Exception (hf_hub_download raises many specific
        # types) and re-raise with a path-specific message.
        _ename = type(_e).__name__
        _emsg = str(_e).split('\n')[0][:200]
        # Map known HF exception names to actionable guidance.
        if _ename == 'RepositoryNotFoundError':
            raise RuntimeError(
                f'REPO NOT FOUND: {repo_id}. Either the repo name is wrong '
                f'or it has been moved/deleted. Check that the URL '
                f'https://huggingface.co/{repo_id} is correct.'
            ) from _e
        if _ename == 'GatedRepoError' or 'gated' in _emsg.lower() or 'restricted' in _emsg.lower():
            raise RuntimeError(
                f'GATED REPO: {repo_id}. You MUST accept the license at\n'
                f'    https://huggingface.co/{repo_id}\n'
                f'before your HF_TOKEN can access it. Steps:\n'
                f'  1. Open the URL above in a browser.\n'
                f'  2. Click "Agree and access repository" at the top.\n'
                f'  3. Re-run STEP 2 - the same HF_TOKEN will then work.\n'
                f'(Your token is 30+ chars and was sent, so the issue '
                f'is license acceptance, not token validity.)'
            ) from _e
        if _ename == 'RemoteEntryNotFoundError' or _ename == 'EntryNotFoundError':
            raise RuntimeError(
                f'FILE NOT FOUND: {filename} in {repo_id}. The file '
                f'doesn\'t exist at that path. Check the actual file '
                f'listing at https://huggingface.co/{repo_id}/tree/main'
                f'{("/" + _subfolder) if _subfolder else ""} for the '
                f'correct filename.'
            ) from _e
        if _ename == 'RevisionNotFoundError':
            raise RuntimeError(
                f'REVISION NOT FOUND: {repo_id}@{_e}. The branch/commit '
                f'doesn\'t exist. Try revision="main".'
            ) from _e
        # Network or unknown - re-raise with original message.
        raise RuntimeError(f'[{_ename}] {_emsg}') from _e



_files_to_fetch = [
    ('Lightricks/LTX-2.5', f'text_encoders/{TEXT_ENCODER}.safetensors', 'text_encoders'),
    (_t_repo, _t_filename, _scope),
    ('Lightricks/LTX-2.5', f'vae/{VIDEO_VAE}.safetensors', 'vae'),
    # Audio VAE goes to models/vae/ - the official Lightricks/ComfyUI-LTXVideo
    # custom node loader reads from there. The stock ComfyUI VAELoader
    # also reads from models/vae/ and correctly detects LTX Audio via
    # 'vocoder.resblocks.0.convs1.0.weight' in sd, applying
    # state_dict_prefix_replace to handle the audio_vae.* / vocoder.*
    # prefixes (comfy/sd.py VAE.__init__).
    ('Lightricks/LTX-2.5', 'vae/ltx-2.5-audio-vae-bf16.safetensors', 'vae'),
]
if USE_UPSCALER:
    _files_to_fetch.append(
        ('Lightricks/LTX-2.5', 'latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors', 'latent_upscale_models')
    )
    _files_to_fetch.append(
        ('Lightricks/LTX-2.5-22b-IC-LoRA-Pixel-Spatial-Upscaler', 'ltx-2.5-22b-ic-lora-pixel-spatial-upscaler-x2-1.0.safetensors', 'loras')
    )

# Build per-repo fetch plans.
# _files_to_fetch entries are: (repo_id, hf_repo_path, comfy_models_subfolder)
_PATTERNS_BY_REPO = {}
for _repo, _fn, _subdir in _files_to_fetch:
    _PATTERNS_BY_REPO.setdefault(_repo, []).append((_fn, _subdir))
print(f'  Fetching from {len(_PATTERNS_BY_REPO)} repo(s) ...')

# Diagnostic: print the actual paths we are about to fetch.
print('  Path sanity check:', flush=True)
for _dr, _dps in _PATTERNS_BY_REPO.items():
    print(f'    {_dr}:', flush=True)
    for _dp, _sc in _dps:
        print(f'      OK: {_dp} -> models/{_sc}/')

if _diag_stale_count > 0:
    print()
    print('  *** STALE CELL SOURCE DETECTED ***', flush=True)
    print(f'  {_diag_stale_count} path(s) have doubled prefixes.', flush=True)
    print('  This happens when Colab serves an older cached cell source.', flush=True)
    print('  Fix: Runtime > Restart session, then re-run STEP 2.', flush=True)
    print('  OR pull the latest commit:', flush=True)
    print('    !cd /content/AEI-Colab-Notebooks && git pull', flush=True)

# One-time migration: replace any existing symlinks under
# COMFY_DIR/models/ with hard copies. The symlinks were created by
# earlier versions of STEP 2 and are incompatible with comfy-aimdo's
# pread() path (EIO at certain offsets through Drive FUSE symlinks).
_migrated = 0
_dangling_removed = 0
if COMFY_DIR.joinpath('models').exists():
    for _model_dir in COMFY_DIR.joinpath('models').iterdir():
        if not _model_dir.is_dir():
            continue
        for _sf in _model_dir.iterdir():
            if _sf.is_symlink():
                try:
                    _target = _sf.resolve()
                    if _target.exists():
                        # Healthy symlink - convert to hard copy
                        # so comfy-aimdo's pread() works.
                        import shutil as _sh
                        _sf.unlink()
                        _sh.copy(_target, _sf)
                        _migrated += 1
                    else:
                        # Dangling symlink - target was deleted/moved.
                        # Remove it so the download code can place a
                        # real file at this path. If we don't, the file
                        # exists (as a broken symlink) but resolves to
                        # nothing - ComfyUI's folder_paths.get_full_path
                        # skips it with 'exists but doesn't link anywhere'.
                        _sf.unlink()
                        _dangling_removed += 1
                except Exception as _mig_err:
                    print(f'    WARN: could not migrate symlink {_sf}: {type(_mig_err).__name__}', flush=True)
if _migrated:
    print(f'  Migrated {_migrated} symlink(s) to hard copies (aimdo compatibility).', flush=True)
if _dangling_removed:
    print(f'  Removed {_dangling_removed} dangling symlink(s) so the next download can land a real file.', flush=True)

t0 = time.time()
for _repo, _patterns in _PATTERNS_BY_REPO.items():
    print(f'    {_repo}: {len(_patterns)} file(s) ...')
    try:
        for _pat, _scope_for_file in _patterns:
            _local = Path(HF_CACHE) / _pat
            _local.parent.mkdir(parents=True, exist_ok=True)
            _downloaded = _download_file(_repo, _pat, _local)
            if _downloaded and Path(_downloaded).exists():
                _comfy_dest = COMFY_DIR / 'models' / _scope_for_file / Path(_pat).name
                _comfy_dest.parent.mkdir(parents=True, exist_ok=True)
                if not _comfy_dest.exists() or _comfy_dest.stat().st_size != Path(_downloaded).stat().st_size:
                    try:
                        import shutil
                        shutil.copy(_downloaded, _comfy_dest)
                        sz_gb = _comfy_dest.stat().st_size / 1024**3
                        print(f'    -> ComfyUI models/{_scope_for_file}/{Path(_pat).name} (copy, {sz_gb:.2f} GB)', flush=True)
                    except Exception as _copy_err:
                        print(f'    !! Hard copy failed for {_comfy_dest}: {type(_copy_err).__name__}: {str(_copy_err)[:200]}', flush=True)
                else:
                    sz_gb = _comfy_dest.stat().st_size / 1024**3
                    print(f'    -> ComfyUI models/{_scope_for_file}/{Path(_pat).name} (copy, {sz_gb:.2f} GB, already cached)', flush=True)
    except Exception as e:
        print(f'    WARN: download from {_repo} failed: {type(e).__name__}: {str(e)[:200]}')
        err_str = str(e)
        # Lightricks/LTX-2.5 is a GATED repo. Without a token, HF returns
        # 401 regardless of file path. Catch 401 first so users see the
        # actionable 'get a token' message rather than the wrong
        # 'file path wrong' 404 message.
        if '401' in err_str or 'Unauthorized' in err_str:
            print(f'    GOT 401 = gated repo, no HF_TOKEN. The {_repo!r} repo is gated -')
            print(f'        your request needs authentication. To fix:')
            print(f'          1. Visit https://huggingface.co/{_repo} and click')
            print(f'             "Agree and access repository" (you only need to do')
            print(f'             this once per repo).')
            print(f'          2. Create a free read-token at https://huggingface.co/settings/tokens')
            print(f'          3. Either paste it into HF_TOKEN_BELOW below, OR add')
            print(f'             it as a Colab secret named HF_TOKEN (Tools > Secrets).')
            print(f'             Then re-run STEP 2 (cache is resumable).')
        elif '403' in err_str or 'Forbidden' in err_str:
            print(f'    GOT 403 = token has wrong scope. Your HF_TOKEN exists but')
            print(f'        is not authorized for {_repo}. Most common cause: license')
            print(f'        not accepted yet. Visit https://huggingface.co/{_repo},')
            print(f'        click "Agree and access repository", then re-run STEP 2.')
        elif '404' in err_str or 'Not Found' in err_str:
            # 404 in HF context also means "gated repo + no token" if the
            # Authorization header was bypassed somewhere. Catch any 404 with
            # a hostile message that covers both cases.
            print(f'    GOT 404 = either the file path is wrong OR the repo is')
            print(f'        gated and no HF_TOKEN was sent (HF serves both as 404/401).')
            print(f'        file path attempted: {repr(_patterns[0]) if _patterns else "<empty>"}')
            print(f'        If this is the first download attempt, you probably need')
            print(f'        an HF_TOKEN - see the 401 instructions above.')
            print(f'        Otherwise check https://huggingface.co/{_repo}/tree/main')
            print(f'        to confirm the actual filename (typo or repo refactor?).')
        else:
            print(f'    Unknown error - check your network connection and try:')
            print(f'        curl -I https://huggingface.co/{_repo}/resolve/main/')
            print(f'      If that works locally but STEP 2 keeps failing, the issue')
            print(f'      might be HF rate limiting on this anonymous IP.')
        print(f'          Re-run STEP 2 to retry (HF cache is resumable).')




# ============================================================
# Re-persist the dropdown values to builtins at the END of STEP 2 too.
# The early export at the top of this cell (line ~97) covers the
# normal case; this end-of-cell block is a fallback for users whose
# mid-cell logic gets modified or extended - the early export stays
# immutable but this one runs only on a clean run to completion.
# ============================================================
import builtins as _b
_b._TRANSFORMER_VAR = TRANSFORMER
_b._QUANT_VAR = QUANT
_b._TEXT_ENCODER_FILE = TEXT_ENCODER_FILE

# Cache-freshness marker: present in the latest commit.
# If Colab is serving stale cell source, this variable will
# be missing when subsequent cells try to introspect it.
_PATH_DOUBLING_FIX_MARKER = True  # noqa - used as a freshness signal

_b._SCOPE_VAR = _scope

# One-time migration: if a previous STEP 2 run placed the audio VAE
# at the WRONG models/checkpoints/ location (because an earlier commit
# thought LTXVAudioVAELoader needed that path), move it back to the
# correct models/vae/ location which stock VAELoader + ComfyUI's
# VAE auto-detection reads from.
_old_audio_vae = COMFY_DIR / "models" / "checkpoints" / "ltx-2.5-audio-vae-bf16.safetensors"
_new_audio_vae = COMFY_DIR / "models" / "vae" / "ltx-2.5-audio-vae-bf16.safetensors"
if _old_audio_vae.exists() and not _new_audio_vae.exists():
    _new_audio_vae.parent.mkdir(parents=True, exist_ok=True)
    try:
        import shutil
        shutil.move(_old_audio_vae, _new_audio_vae)
        print(f"  Migrated audio VAE: {_old_audio_vae} -> {_new_audio_vae}", flush=True)
    except Exception as _e:
        # Some Drive FUSE setups do not support cross-dir move. Fall back
        # to copy + unlink.
        try:
            shutil.copy(_old_audio_vae, _new_audio_vae)
            _old_audio_vae.unlink()
            print(f"  Migrated (copy+unlink) audio VAE: {_old_audio_vae} -> {_new_audio_vae}", flush=True)
        except Exception as _e2:
            print(f"  WARN: could not migrate audio VAE: {type(_e2).__name__}: {str(_e2)[:200]}", flush=True)
            print(f"         If STEP 7 complains, manually move:", flush=True)
            print(f"           !mv {_old_audio_vae} {_new_audio_vae}", flush=True)

print(f"  Final loader config check: TRANSFORMER={TRANSFORMER}, QUANT={QUANT}")


















In [ ]:
#@title STEP 3 — Launch ComfyUI subprocess (--disable-pinned-memory)

"""
Standard AEI-ComfyUI launch — same pattern as MiniMax-H3 notebook.
The --disable-pinned-memory + --fp16-intermediates + --disable-api-nodes
flags are the model-agnostic recipe that lets the 22B LTX-2.5 pipeline
fit in 24 GB Colab tiers without host-RAM OOM.
"""
import os, sys, time, subprocess, urllib.request, urllib.error

COMFY_DIR = Path('/content/drive/MyDrive/AEI_ComfyUI')
COMFY_HOST = '127.0.0.1'
COMFY_PORT = 8188
COMFY_URL = f'http://{COMFY_HOST}:{COMFY_PORT}'

LAUNCH_CMD = [
    sys.executable, 'main.py',
    '--listen', COMFY_HOST,
    '--port', str(COMFY_PORT),
    '--disable-pinned-memory',
    '--fp16-intermediates',
    '--disable-api-nodes',
    # --disable-mmap: forces safetensors to use plain read() instead of
    # memory-mapped IO. mmap'd reads of >1GB safetensors files can fail
    # on Drive FUSE + Colab with EINVAL/EIO. We HARD-COPY files (not
    # symlink) from HF_CACHE to COMFY_DIR/models/ in STEP 2 to keep the
    # path native to Drive FUSE. --disable-mmap is a belt-and-suspenders
    # fallback for any safetensors read path that still uses mmap.
    # We deliberately keep comfy-aimdo ENABLED: it streams the 15 GB
    # transformer in chunks so it doesn't need 22 GB of VRAM at peak.
    # Without aimdo the legacy loader loads the whole model to VRAM
    # and we OOM at sampling (21.32 / 22.03 GB used + sampler buffers).
    '--disable-mmap',
    '--output-directory', str(COMFY_DIR / 'output'),
    '--input-directory', str(COMFY_DIR / 'input'),
]
# Auto-detect compute capability; Turing (sm_75) gets the VRAM-reserving flags.
import torch as _torch_check
_GPU_CC_AUTO = None
if _torch_check.cuda.is_available():
    _gp_auto = _torch_check.cuda.get_device_properties(0)
    _GPU_CC_AUTO = float(f'{_gp_auto.major}.{_gp_auto.minor}')
    if _GPU_CC_AUTO < 8.0:
        LAUNCH_CMD.extend(['--reserve-vram', '0.9'])
        LAUNCH_CMD.append('--fast-disk')
        print(f'  GPU compute capability {_GPU_CC_AUTO} (Turing) detected — '
              'auto-enabled --reserve-vram 0.9 --fast-disk.')
    else:
        print(f'  GPU compute capability {_GPU_CC_AUTO} (Ampere/Ada/Hopper) — '
              'no extra launch flags needed.')
env = os.environ.copy()

# Kill leftover ComfyUI from prior runs (pkill + fuser pattern).
subprocess.run(['pkill', '-9', '-f', f'{COMFY_DIR.name}.*main.py'], check=False)
time.sleep(2)
for _cmd in (['fuser', '-k', f'{COMFY_PORT}/tcp'], ['lsof', '-ti', f':{COMFY_PORT}']):
    try:
        r = subprocess.run(_cmd, capture_output=True, timeout=5, check=False)
        if r.returncode == 0 and (r.stdout or r.stderr):
            time.sleep(2)
            break
    except (FileNotFoundError, subprocess.TimeoutExpired):
        continue

(COMFY_DIR / 'output').mkdir(parents=True, exist_ok=True)
(COMFY_DIR / 'input').mkdir(parents=True, exist_ok=True)

print(f'  Launching: {" ".join(LAUNCH_CMD)}')
log_path = COMFY_DIR / 'comfyui.log'
log_f = open(log_path, 'wb')
proc = subprocess.Popen(
    LAUNCH_CMD,
    cwd=str(COMFY_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=env,
    preexec_fn=os.setsid,
)
print(f'  PID: {proc.pid}, log: {log_path}')

# poll for /system_stats, but check proc.poll() FIRST.
ready = False
deadline = time.time() + 1200  # 20 minutes - comfy-aimdo init + node registry
                                # for ComfyUI 0.32 takes ~12-15 min on L4.
while time.time() < deadline:
    if proc.poll() is not None:
        print(f'  ComfyUI exited early with code {proc.returncode}. Last 40 log lines:')
        with open(log_path) as f:
            for ln in f.read().splitlines()[-40:]:
                print('   ', ln)
        raise SystemExit('ComfyUI failed to start.')
    try:
        with urllib.request.urlopen(COMFY_URL + '/system_stats', timeout=2) as r:
            r.read()
        ready = True
        break
    except (urllib.error.URLError, ConnectionResetError, OSError):
        pass
    time.sleep(2)

if not ready:
    proc.terminate()
    raise SystemExit(f'ComfyUI did not respond within 20 minutes. Tail of log:\n'
                     + '\n'.join(open(log_path).read().splitlines()[-20:]))

# Sanity check: verify all critical nodes are registered
import requests as _req
_TRANSFORMER_VAR = getattr(__import__("builtins"), "_TRANSFORMER_VAR", None)
_WANTS_GGUF = bool(_TRANSFORMER_VAR and "GGUF" in _TRANSFORMER_VAR)
_loader_set = ['UNETLoader', 'VAELoader', 'EmptyLTXVLatentVideo', 'LTXVConditioning', 'CLIPLoader', 'CLIPTextEncode', 'LTXICLoRALoaderModelOnly']
if _WANTS_GGUF:
    _loader_set.append('UnetLoaderGGUF')

_missing_nodes = []
for _node in _loader_set:
    _r = _req.get(f'{COMFY_URL}/object_info/{_node}', timeout=10)
    if _r.status_code != 200:
        _missing_nodes.append(_node)

if _missing_nodes:
    print(f'  ⚠️ WARNING: Missing registered nodes: {_missing_nodes}')
    print('  ComfyUI custom node import logs (last 50 lines):')
    if os.path.exists(log_path):
        with open(log_path) as _lf:
            for _ln in _lf.read().splitlines()[-50:]:
                print('   ', _ln)
else:
    print('  ✓ All core and IC-LoRA nodes are registered and ready!')

import builtins as _builtins
_builtins.AEI_COMFY_PROC = proc
_builtins.AEI_COMFY_URL = COMFY_URL
_builtins.AEI_COMFY_LOG = log_path
_builtins.AEI_COMFY_DIR = COMFY_DIR
print(f'  Stored AEI_COMFY_PROC / AEI_COMFY_URL in builtins.')
print(f'  ComfyUI ready on {COMFY_URL}. Try the GUI at that URL.')






In [ ]:
#@title STEP 4 — (Optional) Gradio UI for LTX-2.5
"""
Interactive Gradio interface for LTX-2.5 Text-to-Video and Image-to-Video.
Also defines and publishes the canonical `_build_workflow` and
`_build_workflow_two_stage` functions to `builtins` so that STEP 4, STEP 6,
STEP 7, and STEP 8.5 can all run independently in any order.
"""
import os, sys, time, json, uuid, re, random, socket, subprocess, urllib.request, urllib.parse, requests, shutil, pathlib, builtins
from pathlib import Path
import gradio as gr
from IPython.display import display, clear_output

COMFY_HOST = '127.0.0.1'
COMFY_PORT = 8188
COMFY_URL = getattr(builtins, 'AEI_COMFY_URL', f'http://{COMFY_HOST}:{COMFY_PORT}')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR', Path('/content/drive/MyDrive/AEI_ComfyUI')))
OUT_DIR = COMFY_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Single source of truth for transformer & text encoder (from STEP 2 dropdowns)
_TRANSFORMER = getattr(builtins, "_TRANSFORMER_VAR", "Lightricks/LTX-2.5 (distilled int8-convrot)")
_QUANT = getattr(builtins, "_QUANT_VAR", "Q4_K_M")
_TEXT_ENCODER_FILE = getattr(builtins, "_TEXT_ENCODER_FILE", "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors")

_SAFETENSORS_FILES = {
    'Lightricks/LTX-2.5 (distilled int8-convrot)': 'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors',
    'Lightricks/LTX-2.5 (distilled nvfp4)': 'ltx-2.5-22b-distilled-transformer-nvfp4.safetensors',
    'guillaume127/LTX-2.5-FP8': 'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors',
}

_IS_GGUF = "GGUF" in str(_TRANSFORMER) and "ChrisColeTech" not in str(_TRANSFORMER)
_IS_CCTECH = "ChrisColeTech" in str(_TRANSFORMER)

if _IS_CCTECH:
    _LOADER_CLASS = 'CCTechUnetLoader'
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT}.gguf'
elif _IS_GGUF:
    _LOADER_CLASS = 'UnetLoaderGGUF'
    _QUANT_FILE = 'Q3_K_S' if _QUANT == 'Q3_K_S' else _QUANT
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT_FILE}.gguf'
else:
    _LOADER_CLASS = 'UNETLoader'
    _UNET_FILENAME = _SAFETENSORS_FILES.get(_TRANSFORMER, 'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors')

# Node registry introspection
_NODE_REGISTRY = None
_NODE_WARNINGS = set()

def _ensure_registry():
    global _NODE_REGISTRY
    if _NODE_REGISTRY is not None:
        return _NODE_REGISTRY
    try:
        r = requests.get(f'{COMFY_URL}/object_info', timeout=10)
        r.raise_for_status()
        _NODE_REGISTRY = r.json()
    except Exception:
        _NODE_REGISTRY = {}
    return _NODE_REGISTRY

def _has_node(class_type):
    return class_type in _ensure_registry()

def _warn_missing(node_name, fallback_name, hint):
    key = f'{node_name}->{fallback_name}'
    if key in _NODE_WARNINGS:
        return
    _NODE_WARNINGS.add(key)
    print(f'  WARN: {node_name} not registered. Falling back to {fallback_name}. {hint}')

def _slug(s, maxlen=40):
    s = re.sub(r'[^a-zA-Z0-9_-]+', '-', s or '').strip('-')
    s = re.sub(r'-+', '-', s)
    return s[:maxlen].strip('-') or 'untitled'

def _frames_for_duration(duration_s, fps):
    n = max(9, int(round(float(duration_s) * float(fps))))
    return min(n, 480)

_SIGMAS_CANONICAL_9 = [1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0]

def _interp_sigmas(target_count, canonical=_SIGMAS_CANONICAL_9):
    n = len(canonical)
    if target_count == n - 1:
        return canonical
    if target_count < n - 1:
        step = (n - 1) / max(1, target_count)
        return [canonical[int(round(i * step))] for i in range(target_count + 1)]
    out = []
    for i in range(target_count + 1):
        t = i * (n - 1) / target_count
        lo = int(t)
        frac = t - lo
        if lo >= n - 1:
            out.append(canonical[-1])
        else:
            out.append(canonical[lo] * (1 - frac) + canonical[lo + 1] * frac)
    return out

def _upload_image(path):
    if not path:
        return None
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'Image not found: {path}')
    with p.open('rb') as f:
        mime = 'image/png' if p.suffix.lower() == '.png' else 'image/jpeg'
        files = {'image': (p.name, f, mime)}
        data = {'type': 'input', 'overwrite': 'true'}
        r = requests.post(f'{COMFY_URL}/upload/image', files=files, data=data, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f'Upload failed: {r.status_code} {r.text[:300]}')
    return r.json()['name']

def _build_workflow(prompt, width, height, duration, steps, seed, cfg,
                   fps=24, first_frame=None, last_frame=None,
                   first_frame_strength=1.0, last_frame_strength=1.0,
                   negative_prompt="pc game, console game, video game, cartoon, childish, ugly",
                   disable_audio=False, ltx_api_key="",
                   filename_prefix=''):
    length = _frames_for_duration(duration, fps)
    p = {}
    p['6'] = {'class_type': _LOADER_CLASS, 'inputs': {'unet_name': _UNET_FILENAME, 'weight_dtype': 'default'}}
    p['27'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-video-vae-conv-bf16.safetensors'}}
    p['28'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-audio-vae-bf16.safetensors'}}

    _cond_clip_node = '387'
    p[_cond_clip_node] = {'class_type': 'CLIPLoader', 'inputs': {
        'clip_name': _TEXT_ENCODER_FILE,
        'type': 'ltxv',
        'device': 'default',
    }}
    if ltx_api_key and _has_node('GemmaAPITextEncode'):
        p['388'] = {'class_type': 'GemmaAPITextEncode', 'inputs': {
            'api_key': ltx_api_key,
            'prompt': prompt,
            'enhance_prompt': True,
            'ckpt_name': _UNET_FILENAME,
        }}
        _cond_positive_node = '388'
    else:
        p['364'] = {'class_type': 'CLIPTextEncode', 'inputs': {
            'clip': [_cond_clip_node, 0],
            'text': prompt,
        }}
        _cond_positive_node = '364'

    p['373'] = {'class_type': 'CLIPTextEncode', 'inputs': {
        'clip': [_cond_clip_node, 0],
        'text': negative_prompt,
    }}
    _cond_negative_node = '373'

    p['10'] = {'class_type': 'LTXVConditioning', 'inputs': {
        'positive': [_cond_positive_node, 0],
        'negative': [_cond_negative_node, 0],
        'frame_rate': fps,
    }}
    p['4'] = {'class_type': 'EmptyLTXVLatentVideo', 'inputs': {
        'width': width, 'height': height, 'length': length, 'batch_size': 1,
    }}
    p['200'] = {'class_type': 'LTXVEmptyLatentAudio', 'inputs': {
        'frames_number': length,
        'frame_rate': fps,
        'batch_size': 1,
        'audio_vae': ['28', 0],
    }}
    p['201'] = {'class_type': 'LTXVConcatAVLatent', 'inputs': {
        'video_latent': ['4', 0],
        'audio_latent': ['200', 0],
    }}

    latent_upstream = ['201', 0]
    cond_upstream_pos = ['10', 0]
    cond_upstream_neg = ['10', 1]
    _next_id = 100

    def _add_node(spec):
        nonlocal _next_id
        nid = str(_next_id)
        _next_id += 1
        p[nid] = spec
        return nid

    chains = []
    if first_frame and not last_frame:
        chains.append((first_frame, 0, first_frame_strength))
    elif last_frame and not first_frame:
        chains.append((last_frame, -1, last_frame_strength))
    elif first_frame and last_frame:
        chains.append((first_frame, 0, first_frame_strength))
        chains.append((last_frame, -1, last_frame_strength))

    for image_var, frame_idx, strength in chains:
        nid_load = _add_node({'class_type': 'LoadImage', 'inputs': {'image': image_var}})
        nid_add = _add_node({'class_type': 'LTXVAddGuide', 'inputs': {
            'positive': cond_upstream_pos,
            'negative': cond_upstream_neg,
            'vae': ['27', 0],
            'latent': latent_upstream,
            'image': [nid_load, 0],
            'frame_idx': frame_idx,
            'strength': strength,
        }})
        cond_upstream_pos = [nid_add, 0]
        cond_upstream_neg = [nid_add, 1]
        latent_upstream = [nid_add, 2]

    p['5'] = {'class_type': 'CFGGuider', 'inputs': {
        'model': ['6', 0],
        'positive': cond_upstream_pos,
        'negative': cond_upstream_neg,
        'cfg': cfg,
    }}
    p['7'] = {'class_type': 'RandomNoise', 'inputs': {'noise_seed': seed}}
    p['8'] = {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler_ancestral'}}

    _sigmas_for_steps = _interp_sigmas(steps, _SIGMAS_CANONICAL_9)
    _sigmas_str = ", ".join(f"{s:.6g}" for s in _sigmas_for_steps)
    p["9"] = {"class_type": "ManualSigmas", "inputs": {"sigmas": _sigmas_str}}

    if _has_node('LTXVNormalizingSampler'):
        p['11'] = {'class_type': 'LTXVNormalizingSampler', 'inputs': {
            'noise': ['7', 0], 'guider': ['5', 0], 'sampler': ['8', 0],
            'sigmas': ['9', 0], 'latent_image': latent_upstream,
            'video_normalization_factors': '1,1,1,1,1,1,1,1',
            'audio_normalization_factors': '1,1,0.25,1,1,0.25,1,1',
        }}
    else:
        _warn_missing('LTXVNormalizingSampler', 'SamplerCustomAdvanced', 'Using standard sampler.')
        p['11'] = {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['7', 0], 'guider': ['5', 0], 'sampler': ['8', 0],
            'sigmas': ['9', 0], 'latent_image': latent_upstream,
        }}

    p['15'] = {'class_type': 'LTXVSeparateAVLatent', 'inputs': {'av_latent': ['11', 0]}}

    if _has_node('LTXVTiledVAEDecode'):
        p['14'] = {'class_type': 'LTXVTiledVAEDecode', 'inputs': {
            'vae': ['27', 0],
            'latents': ['15', 0],
            'horizontal_tiles': 2, 'vertical_tiles': 2,
            'overlap': 32,
            'last_frame_fix': True,
        }}
    else:
        _warn_missing('LTXVTiledVAEDecode', 'VAEDecodeTiled', 'Using standard tiled VAE decode.')
        p['14'] = {'class_type': 'VAEDecodeTiled', 'inputs': {
            'samples': ['15', 0], 'vae': ['27', 0],
            'tile_size': 512, 'overlap': 64,
            'temporal_size': 64, 'temporal_overlap': 8,
        }}

    if disable_audio:
        p['40'] = {'class_type': 'CreateVideo', 'inputs': {'images': ['14', 0], 'fps': fps}}
    else:
        p['13'] = {'class_type': 'LTXVAudioVAEDecode', 'inputs': {'samples': ['15', 1], 'audio_vae': ['28', 0]}}
        p['40'] = {'class_type': 'CreateVideo', 'inputs': {'images': ['14', 0], 'audio': ['13', 0], 'fps': fps}}

    p['92'] = {'class_type': 'SaveVideo', 'inputs': {
        'video': ['40', 0], 'filename_prefix': filename_prefix or 'video/LTX',
        'format': 'auto', 'codec': 'auto',
    }}
    return {'prompt': p}

def _build_workflow_two_stage(prompt, width, height, duration, steps, seed, cfg,
                              fps=24, first_frame=None, last_frame=None,
                              first_frame_strength=0.7, last_frame_strength=1.0,
                              negative_prompt="pc game, console game, video game, cartoon, childish, ugly",
                              disable_audio=False, ltx_api_key="",
                              filename_prefix=''):
    length = _frames_for_duration(duration, fps)
    p = {}
    p['6'] = {'class_type': _LOADER_CLASS, 'inputs': {'unet_name': _UNET_FILENAME, 'weight_dtype': 'default'}}
    p['27'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-video-vae-conv-bf16.safetensors'}}
    p['28'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-audio-vae-bf16.safetensors'}}
    p['500'] = {'class_type': 'LatentUpscaleModelLoader' if _has_node('LatentUpscaleModelLoader') else 'UpscaleModelLoader', 'inputs': {
        'model_name': 'ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
    }}

    _cond_clip_node = '387'
    p[_cond_clip_node] = {'class_type': 'CLIPLoader', 'inputs': {
        'clip_name': _TEXT_ENCODER_FILE,
        'type': 'ltxv',
        'device': 'default',
    }}
    if ltx_api_key and _has_node('GemmaAPITextEncode'):
        p['388'] = {'class_type': 'GemmaAPITextEncode', 'inputs': {
            'api_key': ltx_api_key,
            'prompt': prompt,
            'enhance_prompt': True,
            'ckpt_name': _UNET_FILENAME,
        }}
        _cond_positive_node = '388'
    else:
        p['364'] = {'class_type': 'CLIPTextEncode', 'inputs': {
            'clip': [_cond_clip_node, 0],
            'text': prompt,
        }}
        _cond_positive_node = '364'

    p['373'] = {'class_type': 'CLIPTextEncode', 'inputs': {
        'clip': [_cond_clip_node, 0],
        'text': negative_prompt,
    }}
    _cond_negative_node = '373'

    p['10'] = {'class_type': 'LTXVConditioning', 'inputs': {
        'positive': [_cond_positive_node, 0],
        'negative': [_cond_negative_node, 0],
        'frame_rate': fps,
    }}
    p['4'] = {'class_type': 'EmptyLTXVLatentVideo', 'inputs': {
        'width': width, 'height': height, 'length': length, 'batch_size': 1,
    }}
    p['200'] = {'class_type': 'LTXVEmptyLatentAudio', 'inputs': {
        'frames_number': length,
        'frame_rate': fps,
        'batch_size': 1,
        'audio_vae': ['28', 0],
    }}
    p['201'] = {'class_type': 'LTXVConcatAVLatent', 'inputs': {
        'video_latent': ['4', 0],
        'audio_latent': ['200', 0],
    }}

    stage1_image_node = None
    if first_frame:
        p['210'] = {'class_type': 'LTXVPreprocess', 'inputs': {
            'image': [first_frame, 0],
            'img_compression': 18,
        }}
        p['211'] = {'class_type': 'LTXVImgToVideoInplace', 'inputs': {
            'vae': ['27', 0],
            'image': ['210', 0],
            'latent': ['201', 0],
            'strength': first_frame_strength,
            'bypass': False,
        }}
        stage1_latent_node = '211'
        stage1_image_node = '210'
    else:
        stage1_latent_node = '201'

    p['5'] = {'class_type': 'CFGGuider', 'inputs': {
        'model': ['6', 0],
        'positive': ['10', 0],
        'negative': ['10', 1],
        'cfg': cfg,
    }}
    p['7'] = {'class_type': 'RandomNoise', 'inputs': {'noise_seed': seed}}
    p['8'] = {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler_ancestral'}}
    _sigmas_s1 = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
    p['9'] = {'class_type': 'ManualSigmas', 'inputs': {'sigmas': _sigmas_s1}}

    if _has_node('LTXVNormalizingSampler'):
        p['11'] = {'class_type': 'LTXVNormalizingSampler', 'inputs': {
            'noise': ['7', 0], 'guider': ['5', 0], 'sampler': ['8', 0],
            'sigmas': ['9', 0], 'latent_image': [stage1_latent_node, 0],
            'video_normalization_factors': '1,1,1,1,1,1,1,1',
            'audio_normalization_factors': '1,1,0.25,1,1,0.25,1,1',
        }}
    else:
        p['11'] = {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['7', 0], 'guider': ['5', 0], 'sampler': ['8', 0],
            'sigmas': ['9', 0], 'latent_image': [stage1_latent_node, 0],
        }}

    p['15'] = {'class_type': 'LTXVSeparateAVLatent', 'inputs': {'av_latent': ['11', 0]}}

    p['220'] = {'class_type': 'LTXVLatentUpsampler', 'inputs': {
        'samples': ['15', 0],
        'upscale_model': ['500', 0],
        'vae': ['27', 0],
    }}

    if stage1_image_node:
        p['221'] = {'class_type': 'LTXVImgToVideoInplace', 'inputs': {
            'vae': ['27', 0],
            'image': [stage1_image_node, 0],
            'latent': ['220', 0],
            'strength': 1.0,
            'bypass': False,
        }}
        stage2_video_node = '221'
    else:
        stage2_video_node = '220'

    p['222'] = {'class_type': 'LTXVConcatAVLatent', 'inputs': {
        'video_latent': [stage2_video_node, 0],
        'audio_latent': ['15', 1],
    }}
    p['225'] = {'class_type': 'CFGGuider', 'inputs': {
        'model': ['6', 0],
        'positive': ['10', 0],
        'negative': ['10', 1],
        'cfg': cfg,
    }}
    p['226'] = {'class_type': 'RandomNoise', 'inputs': {'noise_seed': seed + 1}}
    p['227'] = {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler'}}
    _sigmas_s2 = "0.85, 0.725, 0.4219, 0.0"
    p['228'] = {'class_type': 'ManualSigmas', 'inputs': {'sigmas': _sigmas_s2}}

    if _has_node('LTXVNormalizingSampler'):
        p['230'] = {'class_type': 'LTXVNormalizingSampler', 'inputs': {
            'noise': ['226', 0], 'guider': ['225', 0], 'sampler': ['227', 0],
            'sigmas': ['228', 0], 'latent_image': ['222', 0],
            'video_normalization_factors': '1,1,1,1',
            'audio_normalization_factors': '1,1,0.25,1',
        }}
    else:
        p['230'] = {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['226', 0], 'guider': ['225', 0], 'sampler': ['227', 0],
            'sigmas': ['228', 0], 'latent_image': ['222', 0],
        }}

    p['231'] = {'class_type': 'LTXVSeparateAVLatent', 'inputs': {'av_latent': ['230', 0]}}

    if _has_node('LTXVTiledVAEDecode'):
        p['232'] = {'class_type': 'LTXVTiledVAEDecode', 'inputs': {
            'vae': ['27', 0],
            'latents': ['231', 0],
            'horizontal_tiles': 2, 'vertical_tiles': 2,
            'overlap': 64,
            'last_frame_fix': True,
        }}
    else:
        p['232'] = {'class_type': 'VAEDecodeTiled', 'inputs': {
            'samples': ['231', 0], 'vae': ['27', 0],
            'tile_size': 1024, 'overlap': 128,
            'temporal_size': 64, 'temporal_overlap': 8,
        }}

    if disable_audio:
        p['40'] = {'class_type': 'CreateVideo', 'inputs': {'images': ['232', 0], 'fps': fps}}
    else:
        p['13'] = {'class_type': 'LTXVAudioVAEDecode', 'inputs': {'samples': ['231', 1], 'audio_vae': ['28', 0]}}
        p['40'] = {'class_type': 'CreateVideo', 'inputs': {'images': ['232', 0], 'audio': ['13', 0], 'fps': fps}}

    p['92'] = {'class_type': 'SaveVideo', 'inputs': {
        'video': ['40', 0], 'filename_prefix': filename_prefix or 'video/LTX_2stage',
        'format': 'auto', 'codec': 'auto',
    }}
    return {'prompt': p}

# Export all canonical workflow builders to builtins
builtins._build_workflow = _build_workflow
builtins._build_workflow_two_stage = _build_workflow_two_stage
builtins._upload_image = _upload_image
builtins._slug = _slug
builtins._frames_for_duration = _frames_for_duration
builtins._has_node = _has_node
builtins._warn_missing = _warn_missing
builtins._TEXT_ENCODER_FILE = _TEXT_ENCODER_FILE
builtins._LOADER_CLASS = _LOADER_CLASS
builtins._UNET_FILENAME = _UNET_FILENAME
builtins._URL = COMFY_URL
builtins._OUT_DIR = OUT_DIR
builtins._USE_TWO_STAGE = False

CANVASES = {
    "1152 x 640 - 16:9": (1152, 640),
    "960 x 544 - 16:9":  (960, 544),
    "832 x 480 - 16:9":  (832, 480),
    "704 x 400 - 16:9":  (704, 400),
    "576 x 320 - 16:9":  (576, 320),
    "640 x 1152 - 9:16": (640, 1152),
    "544 x 960 - 9:16":  (544, 960),
    "480 x 832 - 9:16":  (480, 832),
    "400 x 704 - 9:16":  (400, 704),
    "320 x 576 - 9:16":  (320, 576),
    "768 x 768 - 1:1":   (768, 768),
    "576 x 576 - 1:1":   (576, 576),
    "768 x 576 - 4:3":   (768, 576),
    "576 x 768 - 3:4":   (576, 768),
    "1344 x 544 - ~21:9":(1344, 544),
    "1024 x 416 - ~21:9":(1024, 416),
}
builtins._CANVASES = CANVASES

def _fmt(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'

def run_ltx_video(prompt, first_frame, last_frame, canvas_label,
                  duration, steps, cfg, fps, seed, randomize_seed,
                  two_stage, disable_audio, negative_prompt,
                  ltx_api_key, use_prompt_enhancer,
                  progress=gr.Progress(track_tqdm=False)):
    if not prompt or not prompt.strip():
        raise gr.Error('Prompt cannot be empty.')
    if randomize_seed or int(seed) <= 0:
        seed = random.randint(1, 2**31 - 1)
        progress(0.0, desc=f'Seed: {seed}')
    seed = int(seed)
    w, h = CANVASES.get(canvas_label, (832, 480))
    length = _frames_for_duration(duration, fps)
    slug = _slug(prompt, maxlen=30)
    ts = int(time.time())
    prefix = f'video/LTX_gradio_{slug}_{w}x{h}_d{int(duration)}s_f{length}_s{int(steps)}_seed{seed}_{ts}'

    ff_name = _upload_image(first_frame) if first_frame else None
    lf_name = _upload_image(last_frame) if last_frame else None

    builder = _build_workflow_two_stage if two_stage else _build_workflow
    wf = builder(
        prompt=prompt, width=w, height=h, duration=duration,
        steps=int(steps), seed=seed, cfg=float(cfg), fps=int(fps),
        first_frame=ff_name, last_frame=lf_name,
        negative_prompt=negative_prompt,
        disable_audio=disable_audio,
        ltx_api_key=ltx_api_key if use_prompt_enhancer else "",
        filename_prefix=prefix,
    )

    client_id = str(uuid.uuid4())
    progress(0.05, desc='Queueing workflow with ComfyUI...')
    r = requests.post(f'{COMFY_URL}/prompt', json={'prompt': wf['prompt'], 'client_id': client_id}, timeout=60)
    if r.status_code != 200:
        raise gr.Error(f'ComfyUI error {r.status_code}: {r.text[:300]}')
    prompt_id = r.json()['prompt_id']
    progress(0.1, desc=f'Queued as {prompt_id[:8]} — sampling...')

    t0 = time.time()
    while True:
        try:
            h_res = requests.get(f'{COMFY_URL}/history/{prompt_id}', timeout=10).json()
        except Exception:
            time.sleep(3)
            continue
        if prompt_id in h_res:
            entry = h_res[prompt_id]
            if entry.get('status', {}).get('completed'):
                break
            if entry.get('status', {}).get('error'):
                raise gr.Error('ComfyUI error: ' + json.dumps(entry['status'].get('messages', []))[:500])
        elapsed = time.time() - t0
        progress(min(0.1 + 0.85 * (elapsed / (steps * 15)), 0.95),
                 desc=f'Generating... {_fmt(elapsed)} elapsed ({(elapsed / max(int(steps),1)):.1f}s/step est.)')
        time.sleep(3)

    elapsed = time.time() - t0
    paths = []
    for node_out in entry.get('outputs', {}).values():
        for v in node_out.get('videos', []):
            fn = v.get('filename')
            if fn:
                local_candidates = [OUT_DIR / fn, OUT_DIR / 'video' / fn]
                for c in local_candidates:
                    if c.exists():
                        paths.append(str(c))
                        break
                else:
                    paths.append(str(local_candidates[0]))
    if not paths:
        raise gr.Error('Workflow completed but video output file not found.')
    video_path = next((p for p in paths if p.endswith(('.mp4', '.webm', '.mov'))), paths[0])
    report_md = f'**Done in {_fmt(elapsed)}** ({w}x{h}, {length} frames @ {fps} fps, {steps} steps, seed {seed})'
    return video_path, report_md

with gr.Blocks(title='LTX-2.5 (ComfyUI)', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎬 LTX-2.5 — Text/Image-to-Video Generation\n\n'
                f'**Model:** `{_UNET_FILENAME}` · **Text Encoder:** `{_TEXT_ENCODER_FILE}` · **ComfyUI:** `{COMFY_URL}`')
    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(
                lines=4,
                value='A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot, shallow depth of field, cinematic 35mm film grain, soft ambient sound.',
                label='Prompt',
                info='Detailed scene description with camera motion and audio hints.'
            )
            with gr.Row():
                first_frame = gr.Image(label='First frame (optional, for I2V)', type='filepath', sources=['upload'])
                last_frame = gr.Image(label='Last frame (optional, for FLF2V)', type='filepath', sources=['upload'])
            negative_prompt = gr.Textbox(
                lines=2,
                value='pc game, console game, video game, cartoon, childish, ugly',
                label='Negative Prompt',
                info='Helps keep model in photorealistic territory.'
            )
            with gr.Row():
                ltx_api_key = gr.Textbox(label='LTX API Key (optional)', type='password', placeholder='Get key at console.ltx.io')
                use_prompt_enhancer = gr.Checkbox(value=False, label='Use LTX API Prompt Enhancer')
        with gr.Column(scale=2):
            canvas_dd = gr.Dropdown(choices=list(CANVASES.keys()), value='832 x 480 - 16:9', label='Resolution / Canvas')
            with gr.Row():
                duration = gr.Slider(minimum=1, maximum=20, value=5, step=1, label='Duration (seconds)')
                fps = gr.Slider(minimum=16, maximum=60, value=24, step=1, label='FPS')
            with gr.Row():
                steps = gr.Slider(minimum=1, maximum=30, value=8, step=1, label='Steps (8 standard)')
                cfg = gr.Slider(minimum=1.0, maximum=7.0, value=1.0, step=0.1, label='CFG')
            with gr.Row():
                seed = gr.Number(value=42, label='Seed (0 = random)', precision=0)
                randomize_seed = gr.Checkbox(value=True, label='Randomize seed')
            with gr.Row():
                two_stage = gr.Checkbox(value=False, label='Two-Stage Distilled (2x spatial upscale)')
                disable_audio = gr.Checkbox(value=False, label='Disable audio decode')
            btn = gr.Button('Generate Video', variant='primary')
            video_out = gr.Video(label='Generated Video + Audio')
            report = gr.Markdown()

    btn.click(
        fn=run_ltx_video,
        inputs=[prompt, first_frame, last_frame, canvas_dd, duration, steps, cfg, fps, seed, randomize_seed,
                two_stage, disable_audio, negative_prompt, ltx_api_key, use_prompt_enhancer],
        outputs=[video_out, report],
        api_name='generate'
    )
    def _welcome():
        return 'LTX-2.5 ComfyUI backend ready. Choose parameters and click Generate.'
    demo.load(_welcome, inputs=None, outputs=[report])

def _is_port_free(port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        s.bind(('127.0.0.1', port))
        return True
    except OSError:
        return False
    finally:
        s.close()

def _kill_port_holders(port):
    for cmd in (['fuser', '-k', f'{port}/tcp'], ['lsof', '-ti', f':{port}']):
        try:
            r = subprocess.run(cmd, capture_output=True, timeout=5, check=False)
            if r.returncode == 0 and (r.stdout or r.stderr):
                time.sleep(2)
                break
        except Exception:
            continue

_GRADIO_PORT = None
for _candidate in range(7860, 7870):
    _kill_port_holders(_candidate)
    if _is_port_free(_candidate):
        _GRADIO_PORT = _candidate
        break
if _GRADIO_PORT is None:
    _GRADIO_PORT = 7860

clear_output()
demo.queue(concurrency_limit=1).launch(share=False, inline=False, prevent_thread_lock=True, server_port=_GRADIO_PORT, quiet=True)
display(demo)
print(f'STEP 4 launched. Gradio UI running on http://127.0.0.1:{_GRADIO_PORT}')


In [ ]:
#@title STEP 5 — Keep-alive + session summary (ComfyUI status)

import time, json, urllib.request, builtins

URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')

import IPython.display
display(IPython.display.Javascript("""
function KeepAlive() { console.log('Colab session kept alive at ' + new Date().toISOString()); }
setInterval(KeepAlive, 60000);
"""))

print('=' * 72)
print('LTX-2.5 / ComfyUI session summary')
print('=' * 72)

try:
    with urllib.request.urlopen(URL + '/system_stats', timeout=5) as r:
        stats = json.loads(r.read())
    devs = stats.get('devices', [])
    for d in devs:
        free  = d.get('vram_free', 0) / 1024**3
        total = d.get('vram_total', 0) / 1024**3
        print(f'  GPU {d.get("name"):<24} {free:5.1f} GB free / {total:5.1f} GB total')
    print(f'  ComfyUI version  : {stats.get("system", {}).get("comfyui_version", "?")}')
    print(f'  Python version   : {stats.get("system", {}).get("python_version", "?")}')
    print(f'  Embedded at      : {URL}')
    print(f'  Log file         : {getattr(builtins, "AEI_COMFY_LOG", "?")}')
    print(f'  Output dir       : {getattr(builtins, "AEI_COMFY_DIR", "?")}/output')
    print()
    print('  Three ways to use:')
    print('    - STEP 6 (quick test): single video via form params, polls /history')
    print('    - STEP 7 (batch): JSON scene list for production runs')
    print('  STEP 8 tails the ComfyUI log if a run stalls.')
except Exception as e:
    print(f'  [WARN] ComfyUI not reachable: {e}')



In [ ]:
#@title STEP 6 — Quick test (single video generation)

"""
Build a single LTX-2.5 t2v workflow and queue it via POST /prompt.
Uses the same _build_workflow pattern as STEP 4 (which is currently
a stub for LTX-2.5; STEP 6 has the full chain). Mirrors the MiniMax-H3
notebook's STEP 6 but with the LTX nodes.
"""

from pathlib import Path

import os
import sys
import time
import json
import uuid
import urllib.request
import urllib.parse
import requests
import shutil
import gc
import re
import random
import builtins
from IPython.display import display, FileLink

# Default loader-class values. The if/elif/else below REPLACES these
# based on the user's TRANSFORMER dropdown value. Defaults are set FIRST
# so even if the cell-source cache in Colab is stale and the
# if/elif/else block has been cleared, _build_workflow still has
# something to reference. _LOADER_CLASS = "UNETLoader" + the safetensors
# filename is the conservative choice - if the user picked a GGUF
# option in STEP 2, they'll get a 400 from ComfyUI (file not in
# models/diffusion_models/) which the user can fix by re-running
# STEP 1 + STEP 2 cleanly.
_LOADER_CLASS = 'UNETLoader'
_UNET_FILENAME = 'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors'
_TEXT_ENCODER_FILE_DEFAULT = 'gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors'


URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR',
                          Path('/content/drive/MyDrive/AEI_ComfyUI')))
OUT_DIR = COMFY_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Single source of truth: STEP 2 sets builtins._TRANSFORMER_VAR and
# builtins._QUANT_VAR from its dropdown widgets. STEPS 6 + 7 read
# them here. If Colab was restarted between STEP 2 and the current
# cell, those are gone - re-run STEP 2 to restore.
_TRANSFORMER = getattr(builtins, "_TRANSFORMER_VAR", None)
_QUANT = getattr(builtins, "_QUANT_VAR", None)
if _TRANSFORMER is None:
    print("  ERROR: builtins._TRANSFORMER_VAR is None - STEP 2 has not been run.\n")
    print("         Re-run STEP 2 to set the model choice, then this cell.\n")
    raise SystemExit(1)
# Gemma4 text encoder filename (from STEP 2 dropdown)
_TEXT_ENCODER_FILE = getattr(builtins, "_TEXT_ENCODER_FILE",
    "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors")

PROMPT = 'A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot, shallow depth of field, cinematic 35mm film grain, soft ambient sound.'  #@param {type:"string"}
IMG_PATH = ''  #@param {type:"string"}
SEED = 42  #@param {type:"integer"}
LENGTH = 97  #@param {type:"slider", min:9, max:201, step:8}
STEPS = 8  #@param {type:"slider", min:1, max:20, step:1}
CFG = 1.0  #@param {type:"slider", min:1.0, max:7.0, step:0.1}
FPS = 24  #@param {type:"slider", min:16, max:60, step:1}
RESOLUTION = '832 x 480'  #@param ["768 x 432", "832 x 480", "960 x 544", "1152 x 640", "1920 x 1080"]

if SEED <= 0:
    import random
    SEED = random.randint(1, 2**31 - 1)
    print(f'  Random seed: {SEED}')

# Resolutions for LTX-2.5 — must be divisible by 32 and >= 256. We use
# the trained-default 768x432 / 832x480 for short clips to keep VRAM
# low; 1920x1080 is the model's training-native resolution but needs
# ~32 GB VRAM.
_CANVASES = {
    '768 x 432':   (768, 432),
    '832 x 480':   (832, 480),
    '960 x 544':   (960, 544),
    '1152 x 640':  (1152, 640),
    '1920 x 1080': (1920, 1080),
}
_width, _height = _CANVASES[RESOLUTION]
_LENGTH = LENGTH  # frames at 24 fps
print(f'  Prompt   : {PROMPT[:60]}...')
print(f'  Canvas   : {_width} x {_height} ({_LENGTH} frames)')
print(f'  Steps    : {STEPS}')
print(f'  CFG      : {CFG}')
print(f'  Seed     : {SEED}')
print()

import uuid

# Reuse STEP 7's canonical workflow builder (now exposed to builtins).
# This keeps STEP 6 in sync with STEP 7 automatically - no more stale
# inline copies that drift when we add new nodes.
_build_workflow = getattr(builtins, '_build_workflow', None)
if _build_workflow is None:
    raise SystemExit("STEP 7 must run before STEP 6 so the shared _build_workflow is in builtins.\n"
                     "Re-run STEP 7 (or restart runtime and run STEP 7 first).")
USE_TWO_STAGE = getattr(builtins, '_USE_TWO_STAGE', False)
if USE_TWO_STAGE:
    _build_workflow_two_stage = getattr(builtins, '_build_workflow_two_stage', None)
    if _build_workflow_two_stage is None:
        raise SystemExit("USE_TWO_STAGE=True but cell 8 didn't expose _build_workflow_two_stage.\n"
                         "Re-run STEP 7 (the two-stage builder is defined there).")
    _builder = _build_workflow_two_stage
    _stage_label = 'two-stage'
else:
    _builder = _build_workflow
    _stage_label = 'single-stage'


_slug = re.sub(r'[^a-zA-Z0-9_\-]+', '-', PROMPT).strip('-')[:40]
ts = int(time.time())
_prefix = (f'video/LTX_quicktest_{_slug}_{_width}x{_height}_f{_LENGTH}'
           f'_s{STEPS}_seed{SEED}_{ts}')

wf = _builder(PROMPT, _width, _height,
                      duration=max(1, int(round(_LENGTH / max(FPS, 1)))),
                      steps=STEPS, seed=SEED, cfg=CFG,
                      fps=FPS, filename_prefix=_prefix)
nodes = wf['prompt']
print(f"  Mode:    {_stage_label}")


t0 = time.time()
r = requests.post(URL + '/prompt',
                  json={'prompt': nodes, 'client_id': str(uuid.uuid4())},
                  timeout=60)
if r.status_code != 200:
    raise SystemExit(f'ComfyUI rejected the workflow: {r.status_code} {r.text[:1000]}')
prompt_id = r.json()['prompt_id']
print(f'  Queued: {prompt_id[:8]} ... polling /history')

# Poll for completion.
while True:
    h = requests.get(f'{URL}/history/{prompt_id}', timeout=10).json()
    if prompt_id in h:
        entry = h[prompt_id]
        if entry.get('status', {}).get('completed'):
            elapsed = time.time() - t0
            print(f'  Done in {elapsed:.0f}s ({(elapsed/STEPS):.1f}s/step est).')
            break
        if entry.get('status', {}).get('error'):
            raise RuntimeError('ComfyUI failed: ' + json.dumps(entry['status'].get('messages', []), indent=2)[:2000])
        time.sleep(3)

# Resolve output file (ComfyUI strips the directory prefix from the
# filename reported in /history).
import re
paths = []
for node_out in entry.get('outputs', {}).values():
    for out in node_out.get('videos', []):
        fn = out.get('filename')
        if not fn:
            continue
        candidates = [OUT_DIR / fn]
        if '/' not in fn:
            candidates.append(OUT_DIR / 'video' / fn)
        local = None
        for c in candidates:
            if c.exists():
                local = c
                break
        if local is None:
            local = candidates[0]
        paths.append(local)
print(f'\n  Outputs ({len(paths)}):')
for p in paths:
    print(f'    {p}')
video_path = next((str(p) for p in paths if str(p).endswith(('.mp4','.webm','.mov'))), str(paths[0]))
print(f'\n  Open: {video_path}')
display(FileLink(video_path))



In [ ]:
#@title STEP 7 — Batch generation from a JSON scene list

"""
Advanced batch processor. Reads a JSON file containing a list of scenes, each with
its own prompt, optional keyframe images, canvas, duration, steps, and seed. The
ComfyUI subprocess is shared across scenes; each scene is a separate workflow.

JSON format (a list of objects). Each scene can mix any of:
  - Text-to-Video: just `prompt`
  - Image-to-Video: `prompt` + `first_frame`
  - First-Last-Frame: `prompt` + `first_frame` + `last_frame`

Example batch with all three patterns:
```json
[
  {
    "prompt": "A red fox in a snowy pine forest at dawn, slow dolly push-in",
    "canvas": "832 x 480 - 16:9",
    "duration": 5,
    "seed": 42
  },
  {
    "prompt": "The fox raises its head and bounds toward camera, snow flying",
    "first_frame": "/content/drive/MyDrive/keyframes/fox_start.png",
    "first_frame_strength": 0.95,
    "canvas": "832 x 480 - 16:9",
    "duration": 5,
    "seed": 43
  },
  {
    "prompt": "The fox curls up and goes to sleep as the sun rises behind",
    "first_frame": "/content/drive/MyDrive/keyframes/fox_start.png",
    "last_frame":  "/content/drive/MyDrive/keyframes/fox_end.png",
    "first_frame_strength": 0.9,
    "last_frame_strength": 0.9,
    "canvas": "832 x 480 - 16:9",
    "duration": 6,
    "seed": 44
  }
]
```

Fields (only `prompt` is required; rest override the DEFAULTS widgets below):
  - prompt:               (required) text description of the scene
  - first_frame:          (optional) path to first-frame image (I2V start)
  - last_frame:           (optional) path to last-frame image (I2V end; negative frame_idx)
  - canvas:               (optional, default DEFAULT_CANVAS) one of the CANVASES keys
  - duration:             (optional, default DEFAULT_DURATION) seconds (1-20)
  - steps:                (optional, default DEFAULT_STEPS) inference steps (1-100)
  - fps:                  (optional, default DEFAULT_FPS) frames per second (16-60)
  - seed:                 (optional, default 0) 0 or negative = random
  - first_frame_strength: (optional, default DEFAULT_FIRST_FRAME_STRENGTH) conditioning strength for first frame, 0.0-1.0
  - last_frame_strength:  (optional, default DEFAULT_LAST_FRAME_STRENGTH) conditioning strength for last frame, 0.0-1.0
  - negative_prompt:      (optional, default DEFAULT_NEGATIVE_PROMPT) suppress this content
  - disable_audio:        (optional, default DISABLE_AUDIO) skip audio decode for this scene
  - use_prompt_enhancer:  (optional, default USE_PROMPT_ENHANCER) send prompt to api.ltx.video for expansion

Image paths can be absolute or relative to the Drive root. PNG/JPG/WebP
all work. For I2V, the image gets resized to the canvas dimensions
before encoding, so any aspect ratio is fine (it'll get cropped/squashed
slightly if it doesn't match).

Frame count = max(9, duration * fps); LTX-2.5 has a hard minimum of 9 frames.
We cap at 480 frames (20s at DEFAULT_FPS=24) to keep VRAM headroom.

If `SKIP_EXISTING` is True, runs resume by checking for existing files at
the scene index. A batch_log.jsonl file is also written for partial-resume
scenarios where the user manually deleted outputs.

If the JSON file does not exist yet, this cell writes a starter template.
"""

from pathlib import Path

import os
import sys
import time
import json
import uuid
import urllib.request
import urllib.parse
import requests
import shutil
import gc
import re
import random
import hashlib
import builtins
from IPython.display import display, FileLink

# Default loader-class values. The if/elif/else below REPLACES these
# based on the user's TRANSFORMER dropdown value. Defaults are set FIRST
# so even if the cell-source cache in Colab is stale and the
# if/elif/else block has been cleared, _build_workflow still has
# something to reference. _LOADER_CLASS = "UNETLoader" + the safetensors
# filename is the conservative choice - if the user picked a GGUF
# option in STEP 2, they'll get a 400 from ComfyUI (file not in
# models/diffusion_models/) which the user can fix by re-running
# STEP 1 + STEP 2 cleanly.
_LOADER_CLASS = 'UNETLoader'
_UNET_FILENAME = 'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors'
_TEXT_ENCODER_FILE_DEFAULT = 'gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors'


URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR',
                          Path('/content/drive/MyDrive/AEI_ComfyUI')))
OUT_DIR = COMFY_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Detect which optional LTX nodes are registered in the running ComfyUI
# subprocess. If a user restarts Colab and skips STEP 1+3 (the install
# + relaunch that picks up new custom nodes), some optional nodes
# may be missing. We probe /object_info once per STEP 7 run and degrade
# gracefully: missing nodes get a stock-ComfyUI equivalent + a clear
# warning so the user can fix it with a STEP 1 + STEP 3 rerun.
_NODE_REGISTRY = None  # populated lazily on first _has_node() call
_NODE_WARNINGS = set()  # one warning per missing node, not per scene


def _ensure_registry():
    global _NODE_REGISTRY
    if _NODE_REGISTRY is not None:
        return _NODE_REGISTRY
    try:
        r = requests.get(f'{URL}/object_info', timeout=10)
        r.raise_for_status()
        _NODE_REGISTRY = r.json()
    except Exception as e:
        print(f'  WARN: could not fetch /object_info ({type(e).__name__}); assuming all nodes present')
        _NODE_REGISTRY = {}
    return _NODE_REGISTRY


def _has_node(class_type):
    return class_type in _ensure_registry()


def _warn_missing(node_name, fallback_name, hint):
    key = f'{node_name}->{fallback_name}'
    if key in _NODE_WARNINGS:
        return
    _NODE_WARNINGS.add(key)
    print(f'  WARN: {node_name} not registered in this ComfyUI. Falling back to {fallback_name}.')
    print(f'        {hint}')

# Single source of truth: STEP 2 sets builtins._TRANSFORMER_VAR and
# builtins._QUANT_VAR from its dropdown widgets. STEPS 6 + 7 read
# them here. If Colab was restarted between STEP 2 and the current
# cell, those are gone - re-run STEP 2 to restore.
_TRANSFORMER = getattr(builtins, "_TRANSFORMER_VAR", None)
_QUANT = getattr(builtins, "_QUANT_VAR", None)
if _TRANSFORMER is None:
    print("  ERROR: builtins._TRANSFORMER_VAR is None - STEP 2 has not been run.\n")
    print("         Re-run STEP 2 to set the model choice, then this cell.\n")
    raise SystemExit(1)
# Gemma4 text encoder filename (from STEP 2 dropdown)
_TEXT_ENCODER_FILE = getattr(builtins, "_TEXT_ENCODER_FILE",
    "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors")


def _fmt(seconds):
    """Format seconds as H:MM:SS or MM:SS.

    Used by all the elapsed-time status prints so they read identically
    whether the clip took 90s (the H3 case) or 1800s (a slow L4 first run).
    The user doesn't have to do unit math.
    """
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'

# --- Runtime shim: if Colab serves a stale cell source that does NOT
# define _fmt (a helper we lost + re-added in commit b2fd72c), this
# try/except defines a fallback inline before the run loop starts.
# Output is diagnostic - if you see "stale cell source", pull commit
# b2fd72c and restart the runtime.
try:
    _fmt(1)
except NameError:
    print("  WARNING: _fmt was undefined - stale cell source. Pull + restart.")
    def _fmt(s):
        s = int(s); h, rem = divmod(s, 3600); m, ss = divmod(rem, 60)
        return f'{h}:{m:02d}:{ss:02d}' if h else f'{m:02d}:{ss:02d}'
    builtins._fmt = _fmt


BATCH_JSON_PATH = '/content/drive/MyDrive/AEI_3D_Cache/LTX-Video-2.5/batch_scenes.json'  #@param {type:"string"}
# === DEFAULTS: per-scene JSON keys override these. Single source of truth
# for batch-wide defaults in the form widgets, with per-scene overrides
# in the JSON.
DEFAULT_DURATION = 5  #@param {type:"slider", min:1, max:20, step:1}
DEFAULT_STEPS    = 8  #@param {type:"slider", min:1, max:100, step:1}
DEFAULT_FPS      = 24  #@param {type:"slider", min:16, max:60, step:1}
DEFAULT_FIRST_FRAME_STRENGTH = 1.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
DEFAULT_LAST_FRAME_STRENGTH  = 1.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
# Lightricks' official LTX-2.5 workflow uses this negative prompt
# automatically. Helps keep the model from drifting into cartoony /
# gamey territory. Override per-scene via batch JSON: "negative_prompt"
DEFAULT_NEGATIVE_PROMPT = "pc game, console game, video game, cartoon, childish, ugly"  #@param {type:"string"}
# DEFAULT_CANVAS dropdown. Label syntax is just a display nicety - the
# CANVASES dict below is the source of truth for (width, height).
DEFAULT_CANVAS = "832 x 480 - 16:9"  #@param ["1152 x 640 - 16:9", "960 x 544 - 16:9", "832 x 480 - 16:9", "704 x 400 - 16:9", "576 x 320 - 16:9", "640 x 1152 - 9:16", "544 x 960 - 9:16", "480 x 832 - 9:16", "400 x 704 - 9:16", "320 x 576 - 9:16", "768 x 768 - 1:1", "576 x 576 - 1:1", "768 x 576 - 4:3", "576 x 768 - 3:4", "1344 x 544 - ~21:9", "1024 x 416 - ~21:9"] {"allow-input": true}
SKIP_EXISTING = True  #@param {type:"boolean"}
DISABLE_AUDIO = False  #@param {type:"boolean"}
# Prompt enhancer via LTX API (free). When enabled AND an API key is
# set, prompts are sent to api.ltx.video for expansion into detailed
# cinematic instructions before encoding. This replaces the local
# CLIP+CLIPTextEncode path with a single network call - no extra
# VRAM, no extra models to download. Get a free key at
# https://console.ltx.io and paste below.
LTX_API_KEY = ""  #@param {type:"string"}
USE_PROMPT_ENHANCER = False  #@param {type:"boolean"}
# Two-stage workflow: Stage 1 runs 8-step distilled at base resolution,
# Stage 2 upscales 2x spatially and runs 3 refinement steps. Produces
# significantly cleaner details than single-stage at the same output
# resolution. Cost: ~25% longer per clip. Only available with USE_UPSCALER
# (the spatial upscaler model must be downloaded in STEP 2).
USE_TWO_STAGE = False  #@param {type:"boolean"}
RESUME_FROM_LOG = True  #@param {type:"boolean"}

CANVASES = {
    # 16:9 landscape
    "1152 x 640 - 16:9":     (1152, 640),
    "960 x 544 - 16:9":      (960, 544),
    "832 x 480 - 16:9":      (832, 480),
    "704 x 400 - 16:9":      (704, 400),
    "576 x 320 - 16:9":      (576, 320),
    # 9:16 portrait
    "640 x 1152 - 9:16":     (640, 1152),
    "544 x 960 - 9:16":      (544, 960),
    "480 x 832 - 9:16":      (480, 832),
    "400 x 704 - 9:16":      (400, 704),
    "320 x 576 - 9:16":      (320, 576),
    # 1:1 square
    "768 x 768 - 1:1":       (768, 768),
    "576 x 576 - 1:1":       (576, 576),
    # 4:3 / 3:4
    "768 x 576 - 4:3":       (768, 576),
    "576 x 768 - 3:4":       (576, 768),
    # Cinematic wide (~21:9)
    "1344 x 544 - ~21:9":    (1344, 544),
    "1024 x 416 - ~21:9":    (1024, 416),
}

json_path = Path(BATCH_JSON_PATH)
if not json_path.exists():
    json_path.parent.mkdir(parents=True, exist_ok=True)
    template = [
        {"prompt": "A red fox in a snowy pine forest at dawn, slow dolly push-in, snow crunching underfoot.",
         "canvas": DEFAULT_CANVAS, "duration": DEFAULT_DURATION,
         "steps": DEFAULT_STEPS, "fps": DEFAULT_FPS, "seed": 42},
        {"prompt": "A busy night market, neon signs reflecting in puddles, sizzling street food, ambient chatter.",
         "canvas": DEFAULT_CANVAS, "duration": DEFAULT_DURATION,
         "steps": DEFAULT_STEPS, "fps": DEFAULT_FPS, "seed": 7},
        {"prompt": "A cellist playing a slow melody in an empty concert hall, warm stage lighting, distant applause at the end.",
         "canvas": DEFAULT_CANVAS, "duration": DEFAULT_DURATION,
         "steps": DEFAULT_STEPS, "fps": DEFAULT_FPS, "seed": 99},
    ]
    json_path.write_text(json.dumps(template, indent=2))
    print(f'Created batch {json_path} with {len(template)} starter scenes.')
    print('Edit it (add first_frame / last_frame / adjust defaults), then re-run STEP 7.')

with json_path.open() as f:
    scenes = json.load(f)
if not isinstance(scenes, list):
    raise SystemExit(f'Expected a JSON list, got {type(scenes).__name__}')
print(f'Loaded {len(scenes)} scene(s) from {json_path}')

# Resume log - append-only JSONL of completed scenes. Used to skip
# already-rendered scenes when RESUME_FROM_LOG is True.
_log_path = json_path.parent / (json_path.stem + '.log.jsonl')
# _completed: set of scene indexes already done.
# _scene_hashes: {scene_idx: prompt_hash} for hash-based change detection.
_completed = set()
_scene_hashes = {}
if RESUME_FROM_LOG and _log_path.exists():
    with _log_path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                if rec.get('status') == 'ok':
                    idx = rec['index']
                    _completed.add(idx)
                    if 'prompt_hash' in rec:
                        _scene_hashes[idx] = rec['prompt_hash']
            except (json.JSONDecodeError, KeyError):
                continue
    if _completed:
        print(f'Resume log: {len(_completed)} scene(s) already completed (will skip).')

def _scene_prompt_hash(sc):
    '''Stable hash of the scene settings that affect output.

    Includes prompt, all override fields, and the effective
    canvas/duration/fps/steps so changing any of these triggers
    a re-run even at the same index.
    '''
    keys = ('prompt', 'canvas', 'duration', 'fps', 'steps',
            'seed', 'first_frame', 'last_frame',
            'first_frame_strength', 'last_frame_strength',
            'negative_prompt', 'disable_audio', 'use_prompt_enhancer')
    blob = '|'.join(f'{k}={sc.get(k, "")}' for k in keys)
    return hashlib.sha256(blob.encode('utf-8')).hexdigest()[:16]


# GGUF loader detection: TRANSFORMER option containing "GGUF" means
# the city96/ComfyUI-GGUF unet path. CCTechUnetLoader is a
# separate custom node installed via ComfyUI Manager.
_IS_GGUF = "GGUF" in _TRANSFORMER and "ChrisColeTech" not in _TRANSFORMER
_IS_CCTECH = "ChrisColeTech" in _TRANSFORMER
_SAFETENSORS_FILES = {
    'Lightricks/LTX-2.5 (distilled int8-convrot)':
        'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors',
    'Lightricks/LTX-2.5 (distilled nvfp4)':
        'ltx-2.5-22b-distilled-transformer-nvfp4.safetensors',
    'guillaume127/LTX-2.5-FP8':
        'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors',
}
if _IS_CCTECH:
    _LOADER_CLASS = 'CCTechUnetLoader'
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT}.gguf'
elif _IS_GGUF:
    _LOADER_CLASS = 'UnetLoaderGGUF'
    _QUANT_FILE = 'Q3_K_S' if _QUANT == 'Q3_K_S' else _QUANT
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT_FILE}.gguf'
else:
    _LOADER_CLASS = 'UNETLoader'
    _UNET_FILENAME = _SAFETENSORS_FILES[_TRANSFORMER]
print(f"  Loader       : {_LOADER_CLASS} (on {_TRANSFORMER})")
print(f"  Transformer  : {_UNET_FILENAME}")

# Pre-flight check: verify the chosen transformer file actually exists
# on disk before we start iterating scenes. Saves a lot of wasted
# time submitting workflows that 400 because the file is missing.
# HF_CACHE is where hf_hub_download stores files (under
# <HF_CACHE>/models--<org>--<repo>/snapshots/...). ComfyUI's UNETLoader
# reads from a SEPARATE directory at COMFY_DIR/models/<scope>/. STEP 2
# symlinks files into both locations. The pre-flight check below tests
# all known locations.
_COMFY_ROOT = Path(getattr(builtins, 'AEI_COMFY_DIR', '/content/drive/MyDrive/AEI_ComfyUI'))
_HF_ROOT = Path(getattr(builtins, 'AEI_HF_CACHE', '/content/drive/MyDrive/AEI_3D_Cache/LTX-Video-2.5'))

print('  === Pre-flight: scanning COMFY_DIR/models/ ===', flush=True)
for _scope_dir in sorted(_COMFY_ROOT.joinpath('models').iterdir() if _COMFY_ROOT.joinpath('models').exists() else []):
    if _scope_dir.is_dir():
        _files = sorted(_scope_dir.glob('*.safetensors')) + sorted(_scope_dir.glob('*.gguf'))
        if _files:
            print(f'    {_scope_dir.name}/:', flush=True)
            for _f in _files:
                _kind = 'broken-symlink' if (_f.is_symlink() and not _f.resolve().exists()) else ('symlink' if _f.is_symlink() else 'file')
                _sz = '?'
                try:
                    _sz = f'{_f.stat().st_size / 1024**3:.2f}GB'
                except Exception:
                    pass
                print(f'      {_f.name}  [{_kind}, {_sz}]', flush=True)
        else:
            print(f'    {_scope_dir.name}/  (empty)', flush=True)
print()
_unet_path_options = [
    # ComfyUI's models/ tree (where the loader actually reads from):
    _COMFY_ROOT / 'models' / 'diffusion_models' / _UNET_FILENAME,
    _COMFY_ROOT / 'models' / 'unet' / _UNET_FILENAME,
    # HF's own cache (where hf_hub_download writes):
    _HF_ROOT / 'diffusion_models' / _UNET_FILENAME,
    _HF_ROOT / 'unet' / _UNET_FILENAME,
    # HF's symlink-style cache layout (HF created these internally).
    _HF_ROOT / 'models--Lightricks--LTX-2.5' / 'snapshots' / 'main' / 'diffusion_models' / _UNET_FILENAME,
    _HF_ROOT / 'models--realrebelai--LTX-2.5_GGUFs' / 'snapshots' / 'main' / _UNET_FILENAME,
]
_vaes_to_check = {
    'video_vae': [
        _COMFY_ROOT / 'models' / 'vae' / 'ltx-2.5-video-vae-conv-bf16.safetensors',
        _HF_ROOT / 'vae' / 'ltx-2.5-video-vae-conv-bf16.safetensors',
        _HF_ROOT / 'models--Lightricks--LTX-2.5' / 'snapshots' / 'main' / 'vae' / 'ltx-2.5-video-vae-conv-bf16.safetensors',
    ],
    'audio_vae': [
        # Stock VAELoader reads from models/vae/ - same as the official
        # Lightricks workflow placement. ComfyUI auto-detects LTX Audio
        # via 'vocoder.resblocks.0.convs1.0.weight' and strips prefixes.
        _COMFY_ROOT / 'models' / 'vae' / 'ltx-2.5-audio-vae-bf16.safetensors',
        # HF's local_dir cache + symlink-style cache as fallbacks
        _HF_ROOT / 'vae' / 'ltx-2.5-audio-vae-bf16.safetensors',
        _HF_ROOT / 'models--Lightricks--LTX-2.5' / 'snapshots' / 'main' / 'vae' / 'ltx-2.5-audio-vae-bf16.safetensors',
        # Older users may have placed it in models/checkpoints/ via
        # Lightricks/ComfyUI-LTXVideo's LTXVAudioVAELoader. Check
        # there too as a courtesy.
        _COMFY_ROOT / 'models' / 'checkpoints' / 'ltx-2.5-audio-vae-bf16.safetensors',
    ],
}
def _is_real(p):
    """Path exists AND either isn't a symlink or its target resolves."""
    if not p.exists():
        return False
    if p.is_symlink() and not p.resolve().exists():
        return False
    return True

_dangling = []
_unlinked = 0
for _scope_dir in ['diffusion_models', 'unet', 'vae', 'text_encoders', 'checkpoints', 'latent_upscale_models']:
    _sd = _COMFY_ROOT / 'models' / _scope_dir
    if not _sd.is_dir():
        continue
    for _f in _sd.iterdir():
        if _f.is_symlink() and not _f.resolve().exists():
            _dangling.append(_f)
            # Inline unlink so the user doesn't have to re-run STEP 2.
            # STEP 2's download will then place a real file here.
            try:
                _f.unlink()
                _unlinked += 1
            except Exception:
                pass
if _dangling:
    print(f'\n  WARNING: {len(_dangling)} dangling symlink(s) found under COMFY_DIR/models/:')
    for _d in _dangling:
        try:
            _tgt = _d.resolve()
        except Exception:
            _tgt = '<unresolvable>'
        print(f'    {_d.name} (target missing: {_tgt})')
    if _unlinked:
        print(f'  Inlined cleanup: unlinked {_unlinked} dangling symlinks so re-running STEP 2 will land real files.')
    else:
        print('  Re-run STEP 2 to clean these up automatically. STEP 2 also has a migration')
        print('  that converts healthy symlinks to hard copies for comfy-aimdo compatibility.')

_missing = []
_found_path = None
for _p in _unet_path_options:
    if _is_real(_p):
        _found_path = _p
        break
if _found_path is None:
    _missing.append(f'    transformer ({_UNET_FILENAME}) - searched:')
    for _p in _unet_path_options:
        _missing.append(f'        - {_p}')
else:
    print(f'  Transformer : {_found_path} ({_found_path.stat().st_size / 1024**3:.2f} GB)')
for _vae_name, _paths in _vaes_to_check.items():
    _vae_found = None
    for _p in _paths:
        if _is_real(_p):
            _vae_found = _p
            break
    if _vae_found:
        print(f'  {_vae_name:11}: {_vae_found} ({_vae_found.stat().st_size / 1024**3:.2f} GB)')
    else:
        _missing.append(f'    {_vae_name} (searched {len(_paths)} locations - none found)')
# Text encoder pre-flight (was missing before - caused dangling symlink bug)
_text_encoder_paths = [
    _COMFY_ROOT / 'models' / 'text_encoders' / _TEXT_ENCODER_FILE,
    _HF_ROOT / 'text_encoders' / _TEXT_ENCODER_FILE,
    _HF_ROOT / 'models--Lightricks--LTX-2.5' / 'snapshots' / 'main' / 'text_encoders' / _TEXT_ENCODER_FILE,
]
_text_encoder_found = None
for _p in _text_encoder_paths:
    if _is_real(_p):
        _text_encoder_found = _p
        break
if _text_encoder_found:
    print(f'  text_enc    : {_text_encoder_found} ({_text_encoder_found.stat().st_size / 1024**3:.2f} GB)')
else:
    _missing.append(f'    text_encoder ({_TEXT_ENCODER_FILE}) - searched: {len(_text_encoder_paths)} locations')



if _missing:
    print('\n  ERROR: missing required files on disk:')
    for _m in _missing:
        print(_m)
    print('\n  Re-run STEP 2 to download them, then re-run this cell.')
    print('  (STEP 2 only downloads what\'s needed for your current')
    print('   TRANSFORMER + VIDEO_VAE dropdown settings.)')
    if not _dangling:
        # Only SystemExit if there are real missing files - dangling
        # symlinks can be fixed by re-running STEP 2.
        raise SystemExit(1)
    else:
        print()
        print('  ABOVE WARNING shows the dangling symlinks - re-run STEP 2 first.')
        raise SystemExit(1)


def _slug(s, maxlen=40):
    """Compress a prompt to a filesystem-safe slug."""
    s = re.sub(r'[^a-zA-Z0-9_-]+', '-', s).strip('-')
    s = re.sub(r'-+', '-', s)
    return s[:maxlen].strip('-') or 'untitled'

def _frames_for_duration(duration_s, fps):
    """Convert seconds + fps to a frame count valid for LTX-2.5 (min 9, max 480)."""
    n = max(9, int(round(duration_s * fps)))
    return min(n, 480)

def _build_workflow(prompt, width, height, duration, steps, seed, cfg,
                   fps=24, first_frame=None, last_frame=None,
                   first_frame_strength=1.0, last_frame_strength=1.0,
                   negative_prompt="pc game, console game, video game, cartoon, childish, ugly",
                   disable_audio=False, ltx_api_key="",
                   filename_prefix=''):
    """Build the LTX-2.5 t2v/i2v workflow with optional first/last frame conditioning.

    When first_frame or last_frame is provided, the workflow inserts:
      LoadImage -> VAE.encode (inside) -> LTXVAddGuide
    between the conditioning chain and the sampler. LTXVAddGuide chains
    sequentially - first frame attaches at frame_idx=0, last frame at
    frame_idx=-1 (negative = counted from end, AddGuide resolves this).
    Empty latent is the input; the chained guide output becomes the
    new latent input to SamplerCustomAdvanced.
    """
    length = _frames_for_duration(duration, fps)
    p = {}
    # Model + VAE loaders
    p['6'] = {'class_type': _LOADER_CLASS, 'inputs': {
        'unet_name': _UNET_FILENAME,
        'weight_dtype': 'default',
    }}
    p['27'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-video-vae-conv-bf16.safetensors'}}
    # Audio VAE LOADER: stock VAELoader works because ComfyUI's VAE
    # auto-detection picks up the LTX Audio type via
    # 'vocoder.resblocks.0.convs1.0.weight' in the safetensors state
    # dict and applies state_dict_prefix_replace for the audio_vae.*
    # and vocoder.* prefixes. The file lives at models/vae/ - same
    # place the official Lightricks workflow puts it.
    p['28'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-audio-vae-bf16.safetensors'}}
    # Text conditioning
    # Text conditioning path. Three options:
    #   (a) USE_PROMPT_ENHANCER + ltx_api_key -> GemmaAPITextEncode
    #       (free LTX API, no local models, fastest)
    #   (b) default local path -> CLIPLoader + CLIPTextEncode
    #       (14 GB gemma4 loads once, then reuses across scenes)
    #
    # We pick (a) if both flags are set AND the custom node is
    # registered; otherwise fall back to (b).
    # Text conditioning path. We ALWAYS need a CLIPLoader for the
    # negative prompt (CFGGuider requires both positive + negative).
    # The positive conditioning comes from either GemmaAPITextEncode
    # (API path, requires LTX_API_KEY form widget above) or from
    # local CLIPTextEncode (the default).
    #
    # API path uses https://console.ltx.io - free, sub-second, no
    # extra model downloads. The custom node is provided by
    # Lightricks/ComfyUI-LTXVideo (installed in STEP 1).
    #
    # CFG=1.0 means the negative has limited runtime effect but
    # CFGGuider still requires both inputs to be wired.
    _cond_clip_node = '387'
    p[_cond_clip_node] = {'class_type': 'CLIPLoader', 'inputs': {
        'clip_name': _TEXT_ENCODER_FILE,
        'type': 'ltxv',
        'device': 'default',
    }}
    if ltx_api_key and USE_PROMPT_ENHANCER:
        if _has_node('GemmaAPITextEncode'):
            # API path - sends prompt to api.ltx.video, gets back pickled
            # CONDITIONING directly. The API node replaces the local
            # positive CLIPTextEncode (no need to encode the prompt
            # ourselves when we got enhanced embeddings back).
            p['388'] = {'class_type': 'GemmaAPITextEncode', 'inputs': {
                'api_key': ltx_api_key,
                'prompt': prompt,
                'enhance_prompt': True,
                'ckpt_name': _UNET_FILENAME,
            }}
            _cond_positive_node = '388'
        else:
            _warn_missing(
                'GemmaAPITextEncode', 'local CLIPTextEncode',
                'Re-run STEP 1 (auto-pulls ComfyUI-LTXVideo) and STEP 3 '
                '(restarts ComfyUI subprocess). The API key is set but the '
                'custom node providing API encoding is not registered.')
            p['364'] = {'class_type': 'CLIPTextEncode', 'inputs': {
                'clip': [_cond_clip_node, 0],
                'text': prompt,
            }}
            _cond_positive_node = '364'
    else:
        # Local CLIP+CLIPTextEncode path - positive from local gemma4.
        p['364'] = {'class_type': 'CLIPTextEncode', 'inputs': {
            'clip': [_cond_clip_node, 0],
            'text': prompt,
        }}
        _cond_positive_node = '364'
    # Negative prompt. The official Lightricks workflow uses
    # "pc game, console game, video game, cartoon, childish, ugly"
    # to suppress cartoony drift.
    p['373'] = {'class_type': 'CLIPTextEncode', 'inputs': {
        'clip': [_cond_clip_node, 0],
        'text': negative_prompt,
    }}
    _cond_negative_node = '373'
    p['10'] = {'class_type': 'LTXVConditioning', 'inputs': {
        'positive': [_cond_positive_node, 0],
        'negative': [_cond_negative_node, 0],
        'frame_rate': fps,
    }}
    # Empty video latent + empty audio latent (joint diffusion requires
    # both). Without LTXVEmptyLatentAudio, the model produces an audio
    # latent with wrong shape (the model's per-channel-stats broadcast
    # at un_normalize fails with 'tensor a (3328) must match tensor b (128)'
    # because the latent dim is mismatched). The audio latent shape is
    # determined by the audio VAE config (mel_bins, hop_length, etc.) -
    # LTXVEmptyLatentAudio reads this from the audio_vae node.
    p['4'] = {'class_type': 'EmptyLTXVLatentVideo', 'inputs': {
        'width': width, 'height': height, 'length': length, 'batch_size': 1,
    }}
    p['200'] = {'class_type': 'LTXVEmptyLatentAudio', 'inputs': {
        'frames_number': length,
        'frame_rate': fps,
        'batch_size': 1,
        'audio_vae': ['28', 0],
    }}
    # Join the two latents into a single NestedTensor for the sampler.
    p['201'] = {'class_type': 'LTXVConcatAVLatent', 'inputs': {
        'video_latent': ['4', 0],
        'audio_latent': ['200', 0],
    }}
    # Optional first/last frame conditioning via LTXVAddGuide
    latent_upstream = ['201', 0]
    cond_upstream_pos = ['10', 0]
    cond_upstream_neg = ['10', 1]  # LTXVConditioning negative output slot
    _next_id = 100
    def _add_node(spec):
        nonlocal _next_id
        nid = str(_next_id)
        _next_id += 1
        p[nid] = spec
        return nid
    chains = []
    if first_frame and not last_frame:
        chains.append((first_frame, 0, first_frame_strength))
    elif last_frame and not first_frame:
        chains.append((last_frame, -1, last_frame_strength))
    elif first_frame and last_frame:
        chains.append((first_frame, 0, first_frame_strength))
        chains.append((last_frame, -1, last_frame_strength))
    for image_var, frame_idx, strength in chains:
        nid_load = _add_node({'class_type': 'LoadImage', 'inputs': {'image': image_var}})
        nid_add = _add_node({'class_type': 'LTXVAddGuide', 'inputs': {
            'positive': cond_upstream_pos,
            'negative': cond_upstream_neg,
            'vae': ['27', 0],
            'latent': latent_upstream,
            'image': [nid_load, 0],
            'frame_idx': frame_idx,
            'strength': strength,
        }})
        # LTXVAddGuide: [0]=positive [1]=negative [2]=latent
        cond_upstream_pos = [nid_add, 0]
        cond_upstream_neg = [nid_add, 1]
        latent_upstream = [nid_add, 2]
    # Sampler chain
    p['5'] = {'class_type': 'CFGGuider', 'inputs': {
        'model': ['6', 0],
        'positive': cond_upstream_pos,
        'negative': cond_upstream_neg,
        'cfg': cfg,
    }}
    p['7'] = {'class_type': 'RandomNoise', 'inputs': {'noise_seed': seed}}
    p['8'] = {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler_ancestral'}}
    # Sigmas: interpolate the canonical 9-sigma distilled schedule to
    # N steps. The 9-sigma curve (from realrebelai T2V_I2V_Single_Stage_
    # Distilled.json) is the post-training target - LTX-2.5 distilled
    # was optimized for this exact curve.
    #
    # For steps=8 (the default) we use the canonical 9-sigma curve
    # unchanged. For higher step counts (12, 16, 25, 30), we linearly
    # interpolate between the original 9 points to produce a denser
    # schedule that the model still tolerates well. Cost: more steps
    # = slower (proportional), but noticeably cleaner details.
    #
    # For steps < 8 (rare), we just subsample the canonical curve.
    _SIGMAS_CANONICAL_9 = [1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0]

    def _interp_sigmas(target_count, canonical):
        n = len(canonical)
        if target_count == n - 1:  # canonical: 9 sigmas -> 8 steps
            return canonical
        if target_count < n - 1:
            # Subsample: take evenly spaced sigmas
            step = (n - 1) / max(1, target_count)
            return [canonical[int(round(i * step))] for i in range(target_count + 1)]
        # Supersample: linear interpolate to N+1 points
        out = []
        for i in range(target_count + 1):
            t = i * (n - 1) / target_count
            lo = int(t)
            frac = t - lo
            if lo >= n - 1:
                out.append(canonical[-1])
            else:
                out.append(canonical[lo] * (1 - frac) + canonical[lo + 1] * frac)
        return out

    _sigmas_for_steps = _interp_sigmas(steps, _SIGMAS_CANONICAL_9)
    _sigmas_str = ", ".join(f"{s:.6g}" for s in _sigmas_for_steps)
    p["9"] = {"class_type": "ManualSigmas", "inputs": {
        "sigmas": _sigmas_str,
    }}
    # Sampler: prefer LTXVNormalizingSampler (Lightricks/ComfyUI-LTXVideo),
    # which normalizes latents during sampling to prevent oversaturation
    # ('overbaking') and audio clipping. Per the docs, this is only safe
    # with the Distilled model + standard 8-step sigmas - both of which
    # we use. Default factors: video stays at 1.0 (off), audio gets
    # 0.25 at sampling steps 2 and 5 (official recommended for joint AV).
    #
    # Falls back to stock SamplerCustomAdvanced if the custom node pack
    # isn't installed/registered (e.g. user restarted Colab and skipped
    # STEP 1+3). The workflow still runs - just without latent
    # normalization, so outputs may be slightly more saturated.
    if _has_node('LTXVNormalizingSampler'):
        p['11'] = {'class_type': 'LTXVNormalizingSampler', 'inputs': {
            'noise': ['7', 0], 'guider': ['5', 0], 'sampler': ['8', 0],
            'sigmas': ['9', 0], 'latent_image': latent_upstream,
            'video_normalization_factors': '1,1,1,1,1,1,1,1',
            'audio_normalization_factors': '1,1,0.25,1,1,0.25,1,1',
        }}
    else:
        _warn_missing(
            'LTXVNormalizingSampler', 'SamplerCustomAdvanced',
            'Re-run STEP 1 (auto-pulls ComfyUI-LTXVideo) and STEP 3 '
            '(restarts ComfyUI subprocess) to pick up the new nodes.')
        p['11'] = {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['7', 0], 'guider': ['5', 0], 'sampler': ['8', 0],
            'sigmas': ['9', 0], 'latent_image': latent_upstream,
        }}



# =============================================================
# AUDIO VAE — fed via stock VAELoader from models/vae/. The official
# Lightricks workflow places ltx-2.5-audio-vae-bf16.safetensors in
# models/vae/ (not models/checkpoints/). ComfyUI's VAE auto-detection
# picks up the LTX Audio type and applies the prefix replacement.
# Audio decode can be skipped per-scene via batch JSON's
# disable_audio=True (or the global DISABLE_AUDIO form widget).
# =============================================================

    # Decode + composite + save.
    #
    # The sampler output (node 11) is a NestedTensor [video, audio]
    # produced by the joint AV model. We split it with the official
    # LTXVSeparateAVLatent (comfy_extras/nodes_lt.py) so downstream
    # decoders get plain tensor inputs.
    #
    # Video decoder: prefer LTXVTiledVAEDecode (Lightricks/ComfyUI-LTXVideo)
    # which tiles spatially - ~40% peak VRAM reduction on long clips
    # (6 GB vs 10 GB at 1280x720, 121 frames). Falls back to stock
    # VAEDecodeTiled if the custom node isn't registered.
    #
    # Audio decoder: stock LTXVAudioVAEDecode (in core ComfyUI since
    # the LTX-Audio PR merged). No fallback needed.
    p['15'] = {'class_type': 'LTXVSeparateAVLatent', 'inputs': {
        'av_latent': ['11', 0],
    }}
    # Output 0 = video_latent, Output 1 = audio_latent.
    if _has_node('LTXVTiledVAEDecode'):
        # 2x2 spatial tiles with 32px overlap gives seamless blends at
        # the boundaries.
        p['14'] = {'class_type': 'LTXVTiledVAEDecode', 'inputs': {
            'vae': ['27', 0],
            'latents': ['15', 0],
            'horizontal_tiles': 2, 'vertical_tiles': 2,
            'overlap': 32,
            'last_frame_fix': True,
        }}
    else:
        _warn_missing(
            'LTXVTiledVAEDecode', 'VAEDecodeTiled',
            'Re-run STEP 1 (auto-pulls ComfyUI-LTXVideo) and STEP 3 '
            '(restarts ComfyUI subprocess) to pick up the new nodes.')
        p['14'] = {'class_type': 'VAEDecodeTiled', 'inputs': {
            'samples': ['15', 0], 'vae': ['27', 0],
            'tile_size': 512, 'overlap': 64,
            'temporal_size': 64, 'temporal_overlap': 8,
        }}
    if disable_audio:
        # Workaround for upstream ComfyUI audio VAE denormalize bug:
        # skip the audio decode entirely. CreateVideo receives no audio
        # input; video is saved silently.
        p['40'] = {'class_type': 'CreateVideo', 'inputs': {
            'images': ['14', 0], 'fps': fps,
        }}
    else:
        # Audio decode - now receives a clean video-only latent from
        # the split node (not a NestedTensor).
        p['13'] = {'class_type': 'LTXVAudioVAEDecode', 'inputs': {
            'samples': ['15', 1], 'audio_vae': ['28', 0],
        }}
        p['40'] = {'class_type': 'CreateVideo', 'inputs': {
            'images': ['14', 0], 'audio': ['13', 0], 'fps': fps,
        }}
    p['92'] = {'class_type': 'SaveVideo', 'inputs': {
        'video': ['40', 0], 'filename_prefix': filename_prefix,
        'format': 'auto', 'codec': 'auto',
    }}
    return {'prompt': p}
def _build_workflow_two_stage(prompt, width, height, duration, steps, seed, cfg,
                              fps=24, first_frame=None, last_frame=None,
                              first_frame_strength=0.7, last_frame_strength=1.0,
                              negative_prompt="pc game, console game, video game, cartoon, childish, ugly",
                              disable_audio=False, ltx_api_key="",
                              filename_prefix=''):
    """Build the LTX-2.5 two-stage distilled workflow (matches Lightricks' official
    example_workflows/2.5/LTX-2.5_T2V_I2V_Two_Stage_Distilled.json).
    Stage 1: 8-step distilled sampler at base resolution (W x H)
    Stage 2: 2x spatial upscale + 3-step refinement using the spatial upscaler
             model (ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors)
    This produces significantly cleaner details than single-stage at the same
    output resolution - it generates low-res structure + motion first (where
    the distilled model is strongest), then upscales and adds high-frequency
    detail.
    Args:
        first_frame: optional I2V conditioning image. Goes through LTXVPreprocess
            (img_compression=18) then LTXVImgToVideoInplace (strength=first_frame_strength).
            Applied at BOTH stages: stage 1 pins frame 0 of the base-resolution
            latent, stage 2 re-bakes the image into the upscaled latent.
        last_frame: not used in the two-stage workflow. Kept in the signature for
            parity with _build_workflow(); passed through but ignored.
        first_frame_strength: img_strength for stage 1 (default 0.7 to allow
            some generative drift). Stage 2 uses 1.0 (re-pins strongly).
        last_frame_strength: ignored (see last_frame).
    Returns: {'prompt': p} dict ready for POST /prompt.
    Notes on cost:
        Stage 1: ~99 s on L4 (same as single-stage 8-step).
        Stage 2: ~25 s on L4 (3 steps at 2x resolution). Net total ~125 s
            per clip vs ~99 s for single-stage. Worth it for ~40% more detail
            at the same resolution, or same detail at 2x resolution.
    """
    length = _frames_for_duration(duration, fps)
    p = {}
    p['6'] = {'class_type': _LOADER_CLASS, 'inputs': {
        'unet_name': _UNET_FILENAME,
        'weight_dtype': 'default',
    }}
    p['27'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-video-vae-conv-bf16.safetensors'}}
    p['28'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-audio-vae-bf16.safetensors'}}
    p['500'] = {'class_type': 'LatentUpscaleModelLoader' if _has_node('LatentUpscaleModelLoader') else 'UpscaleModelLoader', 'inputs': {
        'model_name': 'ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
    }}
    # Text conditioning (same as single-stage)
    _cond_clip_node = '387'
    p[_cond_clip_node] = {'class_type': 'CLIPLoader', 'inputs': {
        'clip_name': _TEXT_ENCODER_FILE,
        'type': 'ltxv',
        'device': 'default',
    }}
    if ltx_api_key and USE_PROMPT_ENHANCER:
        if _has_node('GemmaAPITextEncode'):
            p['388'] = {'class_type': 'GemmaAPITextEncode', 'inputs': {
                'api_key': ltx_api_key,
                'prompt': prompt,
                'enhance_prompt': True,
                'ckpt_name': _UNET_FILENAME,
            }}
            _cond_positive_node = '388'
        else:
            _warn_missing(
                'GemmaAPITextEncode', 'local CLIPTextEncode',
                'Re-run STEP 1 (auto-pulls ComfyUI-LTXVideo) and STEP 3 '
                '(restarts ComfyUI subprocess).')
            p['364'] = {'class_type': 'CLIPTextEncode', 'inputs': {
                'clip': [_cond_clip_node, 0],
                'text': prompt,
            }}
            _cond_positive_node = '364'
    else:
        p['364'] = {'class_type': 'CLIPTextEncode', 'inputs': {
            'clip': [_cond_clip_node, 0],
            'text': prompt,
        }}
        _cond_positive_node = '364'
    p['373'] = {'class_type': 'CLIPTextEncode', 'inputs': {
        'clip': [_cond_clip_node, 0],
        'text': negative_prompt,
    }}
    _cond_negative_node = '373'
    p['10'] = {'class_type': 'LTXVConditioning', 'inputs': {
        'positive': [_cond_positive_node, 0],
        'negative': [_cond_negative_node, 0],
        'frame_rate': fps,
    }}
    # Stage 1: empty latents
    p['4'] = {'class_type': 'EmptyLTXVLatentVideo', 'inputs': {
        'width': width, 'height': height, 'length': length, 'batch_size': 1,
    }}
    p['200'] = {'class_type': 'LTXVEmptyLatentAudio', 'inputs': {
        'frames_number': length,
        'frame_rate': fps,
        'batch_size': 1,
        'audio_vae': ['28', 0],
    }}
    p['201'] = {'class_type': 'LTXVConcatAVLatent', 'inputs': {
        'video_latent': ['4', 0],
        'audio_latent': ['200', 0],
    }}
    # Optional I2V conditioning for stage 1
    stage1_image_node = None
    if first_frame:
        _uploaded_name = _upload_image(first_frame)
        p['210'] = {'class_type': 'LTXVPreprocess', 'inputs': {
            'image': [_uploaded_name, 0],
            'img_compression': 18,
        }}
        p['211'] = {'class_type': 'LTXVImgToVideoInplace', 'inputs': {
            'vae': ['27', 0],
            'image': ['210', 0],
            'latent': ['201', 0],
            'strength': first_frame_strength,
            'bypass': False,
        }}
        stage1_latent_node = '211'
        stage1_image_node = '210'
    else:
        stage1_latent_node = '201'
    # Stage 1: 8-step distilled sampler
    p['5'] = {'class_type': 'CFGGuider', 'inputs': {
        'model': ['6', 0],
        'positive': ['10', 0],
        'negative': ['10', 1],
        'cfg': cfg,
    }}
    p['7'] = {'class_type': 'RandomNoise', 'inputs': {'noise_seed': seed}}
    p['8'] = {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler_ancestral'}}
    _sigmas_s1 = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
    p['9'] = {'class_type': 'ManualSigmas', 'inputs': {'sigmas': _sigmas_s1}}
    if _has_node('LTXVNormalizingSampler'):
        p['11'] = {'class_type': 'LTXVNormalizingSampler', 'inputs': {
            'noise': ['7', 0], 'guider': ['5', 0], 'sampler': ['8', 0],
            'sigmas': ['9', 0], 'latent_image': [stage1_latent_node, 0],
            'video_normalization_factors': '1,1,1,1,1,1,1,1',
            'audio_normalization_factors': '1,1,0.25,1,1,0.25,1,1',
        }}
    else:
        p['11'] = {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['7', 0], 'guider': ['5', 0], 'sampler': ['8', 0],
            'sigmas': ['9', 0], 'latent_image': [stage1_latent_node, 0],
        }}
    p['15'] = {'class_type': 'LTXVSeparateAVLatent', 'inputs': {
        'av_latent': ['11', 0],
    }}
    # Stage 2: 2x upscale + 3-step refinement
    p['220'] = {'class_type': 'LTXVLatentUpsampler', 'inputs': {
        'samples': ['15', 0],
        'upscale_model': ['500', 0],
        'vae': ['27', 0],
    }}
    if stage1_image_node:
        p['221'] = {'class_type': 'LTXVImgToVideoInplace', 'inputs': {
            'vae': ['27', 0],
            'image': [stage1_image_node, 0],
            'latent': ['220', 0],
            'strength': 1.0,
            'bypass': False,
        }}
        stage2_video_node = '221'
    else:
        stage2_video_node = '220'
    p['222'] = {'class_type': 'LTXVConcatAVLatent', 'inputs': {
        'video_latent': [stage2_video_node, 0],
        'audio_latent': ['15', 1],
    }}
    p['225'] = {'class_type': 'CFGGuider', 'inputs': {
        'model': ['6', 0],
        'positive': ['10', 0],
        'negative': ['10', 1],
        'cfg': cfg,
    }}
    p['226'] = {'class_type': 'RandomNoise', 'inputs': {'noise_seed': seed + 1}}
    p['227'] = {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler'}}
    _sigmas_s2 = "0.85, 0.725, 0.4219, 0.0"
    p['228'] = {'class_type': 'ManualSigmas', 'inputs': {'sigmas': _sigmas_s2}}
    if _has_node('LTXVNormalizingSampler'):
        p['230'] = {'class_type': 'LTXVNormalizingSampler', 'inputs': {
            'noise': ['226', 0], 'guider': ['225', 0], 'sampler': ['227', 0],
            'sigmas': ['228', 0], 'latent_image': ['222', 0],
            'video_normalization_factors': '1,1,1,1',
            'audio_normalization_factors': '1,1,0.25,1',
        }}
    else:
        p['230'] = {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['226', 0], 'guider': ['225', 0], 'sampler': ['227', 0],
            'sigmas': ['228', 0], 'latent_image': ['222', 0],
        }}
    # Decode + save
    p['231'] = {'class_type': 'LTXVSeparateAVLatent', 'inputs': {
        'av_latent': ['230', 0],
    }}
    if _has_node('LTXVTiledVAEDecode'):
        p['232'] = {'class_type': 'LTXVTiledVAEDecode', 'inputs': {
            'vae': ['27', 0],
            'latents': ['231', 0],
            'horizontal_tiles': 2, 'vertical_tiles': 2,
            'overlap': 64,
            'last_frame_fix': True,
        }}
    else:
        p['232'] = {'class_type': 'VAEDecodeTiled', 'inputs': {
            'samples': ['231', 0], 'vae': ['27', 0],
            'tile_size': 1024, 'overlap': 128,
            'temporal_size': 64, 'temporal_overlap': 8,
        }}
    if disable_audio:
        p['40'] = {'class_type': 'CreateVideo', 'inputs': {
            'images': ['232', 0], 'fps': fps,
        }}
    else:
        p['13'] = {'class_type': 'LTXVAudioVAEDecode', 'inputs': {
            'samples': ['231', 1], 'audio_vae': ['28', 0],
        }}
        p['40'] = {'class_type': 'CreateVideo', 'inputs': {
            'images': ['232', 0], 'audio': ['13', 0], 'fps': fps,
        }}
    p['92'] = {'class_type': 'SaveVideo', 'inputs': {
        'video': ['40', 0], 'filename_prefix': filename_prefix,
        'format': 'auto', 'codec': 'auto',
    }}
    return {'prompt': p}

def _upload_image(path):
    """Upload a local image to ComfyUI server (returns the server-side filename)."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    with p.open('rb') as f:
        mime = 'image/png' if p.suffix.lower() == '.png' else 'image/jpeg'
        files = {'image': (p.name, f, mime)}
        data = {'type': 'input', 'overwrite': 'true'}
        r = requests.post(f'{URL}/upload/image', files=files, data=data, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f'upload failed: {r.status_code} {r.text[:300]}')
    return r.json()['name']

# Print the node-availability status once, before the loop. This
# makes it obvious which optional LTX nodes are registered in the
# running ComfyUI and which ones we fell back to stock equivalents
# for. A user seeing a fallback warning here knows they need to
# re-run STEP 1 (to pull latest custom node packs) + STEP 3 (to
# restart ComfyUI and re-register nodes).
_OPTIONAL_NODES = [
    ('LTXVNormalizingSampler', 'latent normalization during sampling'),
    ('LTXVTiledVAEDecode',     'spatial-tiled VAE decode (lower VRAM)'),
    ('GemmaAPITextEncode',     'LTX API prompt enhancement'),
    ('LTXVSeparateAVLatent',   'AV latent splitter (core ComfyUI)'),
    ('LTXVEmptyLatentAudio',   'empty audio latent (core ComfyUI)'),
    ('LTXVConcatAVLatent',     'AV latent packer (core ComfyUI)'),
    ('LTXVAudioVAEDecode',     'audio VAE decode (core ComfyUI)'),
    ('LTXVAddGuide',           'I2V conditioning (core ComfyUI)'),
    ('LTXVConditioning',       'text conditioning (core ComfyUI)'),
]
print('  ComfyUI node availability:')
for _node, _desc in _OPTIONAL_NODES:
    _ok = '✓' if _has_node(_node) else '✗'
    print(f'    {_ok}  {_node:30s}  {_desc}')
_missing = [n for n, _ in _OPTIONAL_NODES if not _has_node(n)]
if _missing:
    _core_missing = [n for n in _missing if any(n.startswith(p) for p in ('LTXV',))]
    _core_only = [n for n in _core_missing if n in ('LTXVSeparateAVLatent',
                                                    'LTXVEmptyLatentAudio',
                                                    'LTXVConcatAVLatent',
                                                    'LTXVAudioVAEDecode',
                                                    'LTXVAddGuide',
                                                    'LTXVConditioning')]
    if _core_only:
        print(f'\n  WARNING: core LTX nodes missing. ComfyUI v0.3.2+ required.')
        print(f'           Re-run STEP 1 to update ComfyUI core + STEP 3 to restart.')
    else:
        print(f'\n  Optional nodes missing — workflow will run with fallbacks.')
        print(f'  To pick them up: re-run STEP 1 (auto-pulls custom nodes) + STEP 3 (restart).')

results = []
total_start = time.time()
for i, sc in enumerate(scenes):
    if i in _completed:
        print(f'  [{i+1}/{len(scenes)}] SKIP (resume log)')
        results.append(None)
        continue
    prompt = sc.get('prompt', '').strip()
    if not prompt:
        print(f'  [{i+1}/{len(scenes)}] SKIP: empty prompt')
        results.append(None)
        continue
    canvas_label = sc.get('canvas', DEFAULT_CANVAS)
    w, h = CANVASES.get(canvas_label, (832, 480))
    duration = int(sc.get('duration', DEFAULT_DURATION))
    fps      = int(sc.get('fps',      DEFAULT_FPS))
    steps    = int(sc.get('steps',    DEFAULT_STEPS))
    seed     = int(sc.get('seed',     0))
    if seed <= 0:
        seed = random.randint(1, 2**31 - 1)
    ff_strength = float(sc.get('first_frame_strength', DEFAULT_FIRST_FRAME_STRENGTH))
    lf_strength = float(sc.get('last_frame_strength',  DEFAULT_LAST_FRAME_STRENGTH))
    ff = sc.get('first_frame', '').strip() or None
    lf = sc.get('last_frame',  '').strip() or None
    length = _frames_for_duration(duration, fps)
    scene_slug = _slug(prompt, maxlen=30)
    scene_ts = int(time.time())
    scene_prefix = (f'video/LTX_batch_{i:03d}_{scene_slug}_{w}x{h}_d{duration}s_f{length}'
                    f'_s{steps}_seed{seed}_{scene_ts}')
    # Compute the prompt hash for change detection.
    scene_hash = _scene_prompt_hash(sc)
    # SKIP_EXISTING: skip if (a) resume log says done with same hash,
    # AND (b) the output file still exists on disk.
    if SKIP_EXISTING:
        prev_hash = _scene_hashes.get(i)
        if i in _completed and prev_hash == scene_hash:
            existing = list(OUT_DIR.glob(f'LTX_batch_{i:03d}_*'))
            if existing:
                print(f'  [{i+1}/{len(scenes)}] SKIP (resume log + unchanged: {existing[0].name})')
                results.append(str(existing[0]))
                continue
        elif i in _completed and prev_hash and prev_hash != scene_hash:
            print(f'  [{i+1}/{len(scenes)}] PROMPT CHANGED (hash {prev_hash[:8]} -> {scene_hash[:8]}) - re-running')
        elif i in _completed:
            # Resume log says done but file missing - re-run.
            print(f'  [{i+1}/{len(scenes)}] PROMPT DONE BUT FILE MISSING - re-running')
        # Fall through to render. If SKIP_EXISTING + output file exists
        # without a resume entry, also skip.
        existing = list(OUT_DIR.glob(f'LTX_batch_{i:03d}_*'))
        if existing and i not in _completed:
            print(f'  [{i+1}/{len(scenes)}] SKIP (already exists: {existing[0].name})')
            with _log_path.open('a') as f:
                f.write(json.dumps({'index': i, 'status': 'ok', 'path': str(existing[0]),
                                    'prompt_hash': scene_hash}) + chr(10))
            _completed.add(i)
            results.append(str(existing[0]))
            continue
    # Upload first/last frame images if provided
    try:
        ff_name = _upload_image(ff) if ff else None
        lf_name = _upload_image(lf) if lf else None
    except (FileNotFoundError, RuntimeError) as e:
        print(f'  [{i+1}/{len(scenes)}] FAIL: image upload: {e}')
        results.append(None)
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'image_error', 'error': str(e)}) + chr(10))
        continue
    print(f'\n  [{i+1}/{len(scenes)}] {w}x{h} d{duration}s f{length} {steps}steps fps={fps} seed={seed}')
    print(f'    {prompt[:80]}')
    if ff:
        print(f'    first_frame: {ff} (strength={ff_strength})')
    if lf:
        print(f'    last_frame:  {lf} (strength={lf_strength})')
    cfg = 1.0
    neg = sc.get("negative_prompt", DEFAULT_NEGATIVE_PROMPT)
    # Per-scene audio toggle: 'disable_audio' in batch JSON, falling
    # back to the global DISABLE_AUDIO form widget.
    scene_disable_audio = bool(sc.get("disable_audio", DISABLE_AUDIO))
    # Per-scene prompt-enhancer toggle: 'use_prompt_enhancer' in batch
    # JSON. Falls back to the USE_PROMPT_ENHANCER form widget.
    scene_use_enhancer = bool(sc.get("use_prompt_enhancer", USE_PROMPT_ENHANCER))
    # Per-scene two_stage override. Falls back to the USE_TWO_STAGE widget.
    # Per-scene value of true/false in JSON also accepted (in case the user
    # wants to A/B single vs two-stage within a batch).
    scene_two_stage = bool(sc.get("two_stage", USE_TWO_STAGE))
    if scene_two_stage:
        # Validate the upscaler model is present (downloaded in STEP 2 when
        # USE_UPSCALER=True). Skip the scene with a clear error if not.
        _upscaler_path = COMFY_DIR / "models" / "latent_upscale_models" / "ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"
        if not _upscaler_path.exists():
            print(f'    FAIL: USE_TWO_STAGE=True but upscaler not found at {_upscaler_path}.')
            print(f'           Re-run STEP 2 with USE_UPSCALER=True to download it.')
            results.append(None)
            with _log_path.open('a') as f:
                f.write(json.dumps({'index': i, 'status': 'missing_upscaler'}) + chr(10))
            continue
        _builder = _build_workflow_two_stage
        _stage_label = 'two-stage'
    else:
        _builder = _build_workflow
        _stage_label = 'single-stage'
    wf = _builder(prompt=prompt, width=w, height=h, duration=duration,
                  steps=steps, seed=seed, cfg=cfg, fps=fps,
                  negative_prompt=neg,
                  disable_audio=scene_disable_audio,
                  ltx_api_key=LTX_API_KEY if scene_use_enhancer else "",
                  first_frame=ff_name, last_frame=lf_name,
                  first_frame_strength=ff_strength, last_frame_strength=lf_strength,
                  filename_prefix=scene_prefix)
    print(f'    mode: {_stage_label}')
    nodes = wf['prompt']
    t0 = time.time()
    r = requests.post(f'{URL}/prompt', json={'prompt': nodes, 'client_id': str(uuid.uuid4())}, timeout=60)
    if r.status_code != 200:
        print(f'    FAIL: {r.status_code} {r.text[:300]}')
        results.append(None)
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'submit_error', 'code': r.status_code}) + chr(10))
        continue
    pid = r.json()['prompt_id']
    print(f'    Queued as {pid[:8]} - polling /history for completion ...')
    last_report = 0
    log_path = Path(getattr(builtins, 'AEI_COMFY_LOG', '/content/drive/MyDrive/AEI_ComfyUI/comfyui.log'))
    last_log_size = 0
    if log_path.exists():
        last_log_size = log_path.stat().st_size
    completed_clean = False
    while True:
        try:
            h = requests.get(f'{URL}/history/{pid}', timeout=10).json()
        except (requests.exceptions.RequestException, ValueError):
            time.sleep(5)
            continue
        if pid in h:
            entry = h[pid]
            if entry.get('status', {}).get('completed'):
                elapsed = time.time() - t0
                print(f'    Done in {_fmt(elapsed)} ({(elapsed / steps):.1f}s/step).')
                completed_clean = True
                break
            if entry.get('status', {}).get('error'):
                err = entry['status'].get('messages', [])
                print(f'    FAIL: {json.dumps(err)[:300]}')
                break
        elapsed = time.time() - t0
        # Pulse print: log changes + 5-min heartbeat
        new_lines = []
        if log_path.exists():
            cur_size = log_path.stat().st_size
            if cur_size > last_log_size:
                with log_path.open('rb') as f:
                    f.seek(last_log_size)
                    new_lines = f.read().decode('utf-8', errors='replace').splitlines()
                last_log_size = cur_size
        tail = ' | '.join(new_lines[-3:])[:200] if new_lines else ''
        if tail or time.time() - last_report > 300:
            last_report = time.time()
            print(f'    ... {_fmt(elapsed)} elapsed' + (f'  log: {tail}' if tail else ''), flush=True)
        time.sleep(5)
    if not completed_clean:
        print('    FAIL: polling exited without completion')
        results.append(None)
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'poll_failed'}) + chr(10))
        gc.collect()
        continue
    # Find SaveVideo output and pull locally
    elapsed = time.time() - t0
    local_path = None
    _video_outputs = []
    _all_outputs = {}
    for node_id, node_out in entry.get('outputs', {}).items():
        _all_outputs[node_id] = node_out
        # New ComfyUI API format: outputs[node_id] is a dict with the
        # typed output names (e.g. {'videos': [...], 'audio': [...]}).
        for v in node_out.get('videos', []):
            _video_outputs.append((node_id, v))
    if not _video_outputs:
        # Older ComfyUI format fallback: look at 'output' (singular).
        for node_id, node_out in entry.get('outputs', {}).items():
            for v in node_out.get('output', {}).get('videos', []) if isinstance(node_out.get('output'), dict) else []:
                _video_outputs.append((node_id, v))
    for node_id, v in _video_outputs:
        fn = v['filename']
        local_candidates = [OUT_DIR / fn]
        if '/' not in fn:
            local_candidates.append(OUT_DIR / 'video' / fn)
        for _wait in range(7):
            for c in local_candidates:
                if c.exists():
                    local_path = c
                    break
            if local_path is not None:
                break
            if _wait == 3 and local_path is None:
                for hit in OUT_DIR.glob(f'*/{fn}'):
                    local_path = hit
                    break
            time.sleep(2)
        if local_path is None:
            local_path = local_candidates[0]
            url = f'{URL}/view?filename={urllib.parse.quote(fn)}&type=output'
            try:
                with urllib.request.urlopen(url, timeout=120) as r2, open(local_path, 'wb') as fp:
                    shutil.copyfileobj(r2, fp)
            except urllib.error.HTTPError as e:
                print(f'    WARN: server /view returned {e.code}; falling back to local-only')
        break
    if local_path is not None:
        print(f'    {local_path.name}  ({_fmt(elapsed)})')
        results.append(str(local_path))
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'ok', 'path': str(local_path),
                                'duration_s': elapsed,
                                'prompt_hash': scene_hash}) + chr(10))
    else:
        # Diagnostic: dump the outputs so the user can see what SaveVideo
        # actually produced (or didn't).
        print('    FAIL: no video output from SaveVideo')
        print(f'      Polled nodes: {sorted(_all_outputs.keys())}')
        print(f'      video outputs found: {len(_video_outputs)}')
        for _nid, _no in _all_outputs.items():
            print(f'      node {_nid} outputs keys: {list(_no.keys()) if isinstance(_no, dict) else type(_no).__name__}')
        print(f'      Check {OUT_DIR} directly for the .mp4 file - if SaveVideo')
        print(f'      succeeded, the file may have been written but not exposed')
        print(f'      through the typed output interface.')
        results.append(None)
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'no_output',
                                'debug_outputs': {k: list(v.keys()) if isinstance(v, dict) else str(v)[:200]
                                                  for k, v in _all_outputs.items()}}) + chr(10))
    gc.collect()

total = time.time() - total_start
ok_count = sum(1 for r in results if r)
fail_count = len(results) - ok_count

# Collect per-scene timings from the log (if available)
_per_scene_times = {}
if _log_path.exists():
    with _log_path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                if 'duration_s' in rec:
                    _per_scene_times[rec['index']] = rec['duration_s']
            except (json.JSONDecodeError, KeyError):
                continue

# Sort by index for a clean per-clip table
_sorted = sorted(_per_scene_times.items())

# Compute slowest/fastest
if _sorted:
    _times_only = [t for _, t in _sorted]
    _fastest = min(_times_only)
    _slowest = max(_times_only)
    _median = sorted(_times_only)[len(_times_only) // 2]
    _fastest_idx = next(i for i, t in _sorted if t == _fastest)
    _slowest_idx = next(i for i, t in _sorted if t == _slowest)
else:
    _fastest = _slowest = _median = 0
    _fastest_idx = _slowest_idx = -1

print()
print('=' * 72)
print(f'  BATCH SUMMARY')
print('=' * 72)
print(f'  Scenes:    {ok_count} succeeded, {fail_count} failed, {len(results)} total')
print(f'  Total:     {_fmt(total)}')
if ok_count > 0:
    print(f'  Per clip:  avg {total/ok_count:.1f}s, '
          f'median {_median:.1f}s, '
          f'fastest {_fastest:.1f}s (scene {_fastest_idx + 1}), '
          f'slowest {_slowest:.1f}s (scene {_slowest_idx + 1})')
print(f'  Log:       {_log_path}')
print()
print('  Per-clip timings:')
for _idx, _t in _sorted:
    _status = 'OK ' if _idx in _completed else 'FAIL'
    print(f'    [{_idx + 1:3d}] {_status}  {_fmt(_t)}  ({_t:.1f}s)')
print()
print('  Output files:')
for p in [r for r in results if r]:
    print(f'    {p}')
print('=' * 72)










































# Expose _build_workflow, _build_workflow_two_stage, and helpers to builtins
# so STEP 6 (Quick test) and STEP 8.5 (Smoke test) can reuse the canonical
# workflow builders without duplicating the code. Single source of truth.
import builtins as _b
_b._build_workflow = _build_workflow
_b._build_workflow_two_stage = _build_workflow_two_stage
_b._has_node = _has_node
_b._warn_missing = _warn_missing
_b._TEXT_ENCODER_FILE = _TEXT_ENCODER_FILE
_b._LOADER_CLASS = _LOADER_CLASS
_b._UNET_FILENAME = _UNET_FILENAME
_b._URL = URL
_b._OUT_DIR = OUT_DIR
_b._USE_TWO_STAGE = USE_TWO_STAGE
print(f'  Exposed _build_workflow + _build_workflow_two_stage + helpers to builtins.')


In [ ]:
#@title STEP 8 — Tail ComfyUI log (debugging aid)

import time
from pathlib import Path

LOG = Path(getattr(__import__('builtins'), 'AEI_COMFY_LOG',
                     '/content/drive/MyDrive/AEI_ComfyUI/comfyui.log'))
TAIL_LINES = 80  #@param {type:"slider", min:10, max:500, step:10}

if LOG.exists():
    lines = LOG.read_text(errors='replace').splitlines()
    print(f'  Last {min(TAIL_LINES, len(lines))} of {len(lines)} log lines from {LOG}:')
    print()
    for line in lines[-TAIL_LINES:]:
        print(f'    {line}')
else:
    print(f'  No log file at {LOG}')



In [ ]:
#@title STEP 8.5 — Smoke-test the workflow wiring

"""
Run a minimal t2v workflow at LENGTH=9, STEPS=2 with a tiny canvas.
Verifies the UNETLoader + VAELoader + LTXVConditioning chain wires
up correctly. Should complete in ~30-60 s on L4 (dominated by
model load time, not the denoise itself).
"""
import json, time, uuid, requests, builtins
from pathlib import Path
URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR',
                          Path('/content/drive/MyDrive/AEI_ComfyUI')))
OUT_DIR = COMFY_DIR / 'output'

w, h, length, steps, cfg = 256, 256, 9, 2, 1.0

# _LOADER_CLASS and _UNET_FILENAME are exposed to builtins by STEP 7.
# _build_workflow reads them automatically - we don't need to pull
# them here. The smoke-test still uses the user's chosen model.
seed = 12345

# Minimal workflow (matches Lightricks example_workflows/2.5/).
# Reuse STEP 7's canonical workflow builder. We override steps=2
# and length=9 to make this a fast smoke-test (still goes through
# the full joint AV chain - just with the smallest meaningful inputs).
_build_workflow = getattr(builtins, '_build_workflow', None)
if _build_workflow is None:
    raise SystemExit("STEP 7 must run before STEP 8.5 so the shared _build_workflow is in builtins.\n"
                     "Re-run STEP 7 (or restart runtime and run STEP 7 first).")
# Smoke test uses single-stage (faster, more diagnostic).
# The two-stage path would push the L4 closer to OOM at this minimal size.
USE_TWO_STAGE = getattr(builtins, '_USE_TWO_STAGE', False)
_builder = _build_workflow  # single-stage by default

wf = _builder(
    prompt="A fluffy white cloud drifting across a blue sky.",
    width=w, height=h,
    duration=max(1, int(round(length / 24))),
    steps=steps, seed=seed, cfg=cfg,
    fps=24,
    filename_prefix=f"smoketest/LTX_{w}x{h}_s{steps}_{int(time.time())}",
)
nodes = wf["prompt"]

print(f'  POST /prompt: {w}x{h}, length={length}, {steps} steps ...')
r = requests.post(URL + '/prompt',
                  json={'prompt': nodes, 'client_id': str(uuid.uuid4())},
                  timeout=60)
if r.status_code != 200:
    raise SystemExit(f'Workflow rejected: {r.status_code}\n{r.text[:1000]}')
pid = r.json()['prompt_id']
print(f'  Queued as {pid[:8]}; polling /history for completion ...')
t0 = time.time()
while True:
    h = requests.get(f'{URL}/history/{pid}', timeout=10).json()
    if pid in h:
        entry = h[pid]
        if entry.get('status', {}).get('completed'):
            print(f'  Done in {time.time() - t0:.0f}s.')
            for node_out in entry.get('outputs', {}).values():
                for out in node_out.get('videos', []):
                    fn = out.get('filename')
                    if fn:
                        print(f'  Output: {OUT_DIR / fn}')
            break
        if entry.get('status', {}).get('error'):
            print('  FAIL:')
            for msg in entry['status'].get('messages', []):
                print(f'    {msg}')
            break
    time.sleep(3)



In [ ]:
#@title STEP 9 — ReDetail Video Upscaler (Scene-Cut Chunking + 8n+1 Frame Grid + Lossless Audio)
"""
ReDetail — Generative Video Upscaling with LTX-2.5 (Bambushu/redetail)

Why ReDetail fixes previous upscaling issues:
  1. Scene-Cut Chunking: Uses FFmpeg scene detection to place chunk boundaries
     on real camera transitions so no texture seams appear mid-shot.
  2. Strict 8n+1 Frame Arithmetic: LTX-2.5 causal VAE quantizes down to 8n+1.
     ReDetail pads reads to 8n+1 and trims outputs back so 0 frames are dropped
     and audio stays 100% in sync across any video length.
  3. Divisible by 64 Geometry: Solves exact pixel dimensions on the /64 grid.
  4. Pre-Cached Conditioning: Saves 5.6 GB VRAM by skipping Gemma text encoder,
     letting you render larger chunks much faster.
  5. Lossless Full-Length Audio Remuxing: Restores pristine original audio.
"""

import os
import sys
import time
import subprocess
import re
import builtins
from pathlib import Path

URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188').rstrip('/')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR',
                          Path('/content/drive/MyDrive/AEI_ComfyUI')))
REDETAIL_PY = COMFY_DIR / 'redetail' / 'redetail.py'

if not REDETAIL_PY.exists():
    raise SystemExit(f'ReDetail script not found at {REDETAIL_PY}. Please re-run STEP 1.')

OUT_DIR = COMFY_DIR / 'output' / 'upscaled'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============== FORM WIDGETS ==============
SOURCE_VIDEO_PATH = '/content/drive/MyDrive/AEI_3D_Out/LTX-Video-2.5/your_clip.mp4'  #@param {type:"string"}
SCALE_MODE = '1.5 (sweet spot - faster)'  #@param ['1.5 (sweet spot - faster)', '2.0 (2x heavy detail)', '1.0 (re-detail in place)']
VRAM_BUDGET = 350  #@param {type:"integer"}
SCENE_CUT_THRESHOLD = 0.30  #@param {type:"slider", min:0.1, max:0.8, step:0.05}
USE_CACHED_COND = True  #@param {type:"boolean"}
OUTPUT_NAME = 'ReDetail_Upscale'  #@param {type:"string"}

# Parse scale float
if '1.0' in SCALE_MODE:
    SCALE = '1.0'
elif '2.0' in SCALE_MODE:
    SCALE = '2.0'
else:
    SCALE = '1.5'

src = Path(SOURCE_VIDEO_PATH)
if not src.exists():
    raise SystemExit(f'Source video not found: {src}. Edit SOURCE_VIDEO_PATH and re-run.')
if src.suffix.lower() not in ('.mp4', '.mov', '.webm', '.mkv', '.avi'):
    raise SystemExit(f'Unsupported format: {src.suffix}. Please provide an MP4 or MOV file.')

# Ensure VAE aliases exist in models/vae (ReDetail expects ltx-2.5-video-vae-bf16.safetensors)
vae_dir = COMFY_DIR / 'models' / 'vae'
if vae_dir.exists():
    conv_vae = vae_dir / 'ltx-2.5-video-vae-conv-bf16.safetensors'
    std_vae = vae_dir / 'ltx-2.5-video-vae-bf16.safetensors'
    if conv_vae.exists() and not std_vae.exists():
        try:
            std_vae.symlink_to(conv_vae)
        except Exception:
            import shutil
            shutil.copy2(conv_vae, std_vae)
        print(f'  Mapped VAE: {std_vae.name} -> {conv_vae.name}')

# Check for existing rendered chunks from prior runs of this clip to enable instant resume
existing_chunk_dirs = sorted(OUT_DIR.glob(f'{OUTPUT_NAME}_{SCALE}x_{src.stem}_*_chunks'),
                             key=os.path.getmtime, reverse=True)
if existing_chunk_dirs and any(existing_chunk_dirs[0].glob('out_rd_*.mp4')):
    latest_work = existing_chunk_dirs[0]
    ts_match = re.search(r'_(\d+)_chunks$', latest_work.name)
    if ts_match:
        ts = int(ts_match.group(1))
        print(f'  Found existing rendered chunks from prior run: {latest_work.name}')
        print('  ⚡ Instant Resume enabled — will finalize and mux in ~10 seconds!')
    else:
        ts = int(time.time())
else:
    ts = int(time.time())

final_mp4 = OUT_DIR / f'{OUTPUT_NAME}_{SCALE}x_{src.stem}_{ts}.mp4'

# Patch redetail.py for ffmpeg compatibility (-vsync 0 instead of -fps_mode) and instant chunk resume
if REDETAIL_PY.exists():
    _rtext = REDETAIL_PY.read_text(encoding='utf-8')
    # 1. Replace -fps_mode passthrough with -vsync 0 for universal ffmpeg support
    _rtext = _rtext.replace('"-fps_mode", "passthrough"', '"-vsync", "0"')
    _rtext = _rtext.replace("'-fps_mode', 'passthrough'", "'-vsync', '0'")
    
    # 2. Add smart chunk resume to avoid re-rendering existing chunks
    _old_chunk_start = 't0 = time.time()\n        print(f"  chunk {i+1}/{len(segs)}  {L} frames ...", end="", flush=True)'
    _new_chunk_start = '''dst = mine(f"{work}/out_{tag}.mp4")
        if os.path.exists(dst) and nframes(dst) >= L:
            print(f"  chunk {i+1}/{len(segs)}  {L} frames ... (reusing existing rendered chunk) ✓", flush=True)
            if nframes(dst) != L:
                t = mine(dst.replace(".mp4", "_t.mp4"))
                subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", dst, "-frames:v", str(L),
                                "-c:v", "libx264", "-crf", "15", "-pix_fmt", "yuv420p",
                                "-c:a", "copy", t], check=True)
                os.replace(t, dst)
            parts.append(dst)
            continue

        t0 = time.time()
        print(f"  chunk {i+1}/{len(segs)}  {L} frames ...", end="", flush=True)'''

    if _old_chunk_start in _rtext and 'reusing existing rendered chunk' not in _rtext:
        _rtext = _rtext.replace(_old_chunk_start, _new_chunk_start)
    
    REDETAIL_PY.write_text(_rtext, encoding='utf-8')
    print('  ✓ ReDetail engine patched: universal FFmpeg compatibility + instant chunk resume enabled.')

# Build CLI command
cmd = [
    sys.executable, str(REDETAIL_PY),
    str(src),
    '--scale', str(SCALE),
    '--budget', str(VRAM_BUDGET),
    '--comfy', str(URL),
    '--out', str(final_mp4),
]

if USE_CACHED_COND:
    cmd += ['--cached-cond', 'redetail']

# Check if using GGUF transformer
_transformer_choice = getattr(builtins, '_TRANSFORMER_VAR', '')
if 'GGUF' in _transformer_choice:
    _quant = getattr(builtins, '_QUANT_VAR', 'Q4_K_M')
    _gguf_name = f'LTX-2.5-Distilled-{_quant}.gguf'
    cmd += ['--gguf', _gguf_name]

print('='*72)
print(f'ReDetail LTX-2.5 Video Upscaler (Target: {SCALE}x)')
print('='*72)
print(f'  Input clip  : {src.name}')
print(f'  Output target: {final_mp4}')
print(f'  Command     : {" ".join(cmd)}')
print('='*72)
print()

# Stream output in real time
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in iter(proc.stdout.readline, ''):
    if line:
        print(line, end='', flush=True)

proc.stdout.close()
ret = proc.wait()

if ret != 0 or not final_mp4.exists():
    raise SystemExit(f'\nReDetail exited with code {ret}. Check logs above for details.')

print()
print('='*72)
print('RE-DETAIL UPSCALING COMPLETE!')
print('='*72)
print(f'  Output file: {final_mp4}')
print(f'  File size  : {final_mp4.stat().st_size / (1024*1024):.1f} MB')

try:
    from IPython.display import display, FileLink
    display(FileLink(str(final_mp4)))
except Exception:
    pass
